# Roadster-inspired 3D Reinforcement Learning: Kickflip and Ramp Recovery

This notebook is a physically consistent 6-DOF rigid-body simulation of two Roadster-style maneuvers with a configurable 10-nozzle thruster model, a Gymnasium environment, a PPO implementation, a classical reference controller, validation tests, reward-exploit tests, and true 3D rendering of the simulated state.

**Honesty notes (please keep when sharing)**
- Inspired by public coverage of the Tesla Roadster demonstration. This does not imply that Tesla uses reinforcement learning or PPO, and nothing here recovers Tesla's actuators or controller.
- All vehicle numbers other than a few reported event figures are illustrative assumptions. The nozzle layout (called L1) is hypothetical. Gas performance (Isp), valve lag, landing-gear travel and contact parameters are assumptions.
- Reference motions come from my own frame-by-frame reading of two uploaded clips (monocular night footage). Angles are rough, not telemetry.
- Learned results come from three clearly separated sources: (1) the scripted classical controller (not learned), (2) behaviour cloning plus DAgger from that controller, optionally fine-tuned with PPO, (3) pure PPO from scratch. Never mix them up when quoting numbers.
- This notebook file itself has not been executed in Colab. The same code was run in a sandbox with Python 3.12, NumPy only for learning (no deep learning framework).
- Citations, if you add any, should be double checked: I have no database access.

In [ ]:
RUN_MODE = "eval"      # "eval": use the shipped checkpoints.  "train": retrain (BC+DAgger, PPO fine-tune), takes tens of minutes on CPU.
N_EVAL = 50            # held-out episodes per evaluation in this notebook (the full 100-episode tables are in the shipped results JSONs)

In [ ]:
!pip -q install gymnasium imageio-ffmpeg pillow pandas scipy matplotlib

In [ ]:
%%writefile sim3d.py
"""
Roadster-inspired 3D rigid-body simulator (6-DOF) with a configurable nozzle table.

Conventions
  body +X front, +Y left, +Z roof;  world +Z up;  quaternion q = (w,x,y,z) maps body -> world.
Nozzle convention (IMPORTANT)
  Each nozzle stores its EXHAUST direction e_i (where the gas plume goes, body frame).
  The force on the vehicle is the REACTION:  F_i = -f_i * e_i   (f_i >= 0).
  Torque about the centre of mass:           T_i =  r_i x F_i.
  A -Z exhaust (plume pointing at the floor) therefore pushes the car along +Z (up when upright).
All numbers below are ILLUSTRATIVE assumptions, not verified Tesla values.
"""
import math
from dataclasses import dataclass, field
import numpy as np

G0 = 9.80665

# ---------------------------------------------------------------- quaternions
def qmul(a, b):
    aw, ax, ay, az = a; bw, bx, by, bz = b
    return np.array([aw*bw - ax*bx - ay*by - az*bz,
                     aw*bx + ax*bw + ay*bz - az*by,
                     aw*by - ax*bz + ay*bw + az*bx,
                     aw*bz + ax*by - ay*bx + az*bw])

def qconj(q):
    return np.array([q[0], -q[1], -q[2], -q[3]])

def qnorm(q):
    return q / math.sqrt(q[0]*q[0] + q[1]*q[1] + q[2]*q[2] + q[3]*q[3])

def q_axis_angle(axis, ang):
    axis = np.asarray(axis, float)
    n = np.linalg.norm(axis)
    if n < 1e-12:
        return np.array([1.0, 0, 0, 0])
    s = math.sin(0.5*ang) / n
    return np.array([math.cos(0.5*ang), axis[0]*s, axis[1]*s, axis[2]*s])

def q_expmap(rv):
    """quaternion of rotation vector rv (rad)"""
    th = math.sqrt(rv[0]*rv[0] + rv[1]*rv[1] + rv[2]*rv[2])
    if th < 1e-9:
        return np.array([1.0, 0.5*rv[0], 0.5*rv[1], 0.5*rv[2]])
    s = math.sin(0.5*th) / th
    return np.array([math.cos(0.5*th), rv[0]*s, rv[1]*s, rv[2]*s])

def q_to_R(q):
    w, x, y, z = q
    return np.array([[1 - 2*(y*y + z*z), 2*(x*y - w*z),     2*(x*z + w*y)],
                     [2*(x*y + w*z),     1 - 2*(x*x + z*z), 2*(y*z - w*x)],
                     [2*(x*z - w*y),     2*(y*z + w*x),     1 - 2*(x*x + y*y)]])

def q_from_euler_zyx(yaw, pitch, roll):
    """for initial conditions / logging only (intrinsic z-y-x)"""
    return qmul(qmul(q_axis_angle([0, 0, 1], yaw), q_axis_angle([0, 1, 0], pitch)),
                q_axis_angle([1, 0, 0], roll))

def q_err_vec(q, q_ref):
    """rotation vector (body frame) that takes q to q_ref, shortest way"""
    qe = qmul(qconj(q), q_ref)
    if qe[0] < 0:
        qe = -qe
    s = math.sqrt(qe[1]**2 + qe[2]**2 + qe[3]**2)
    if s < 1e-9:
        return 2*qe[1:4]
    ang = 2*math.atan2(s, qe[0])
    return qe[1:4] * (ang / s)

def geodesic(q, q_ref):
    d = abs(float(np.dot(q, q_ref)))
    return 2*math.acos(min(1.0, d))

def tilt_angle(q):
    """angle between body +Z and world +Z (0 = upright, pi = inverted)"""
    R = q_to_R(q)
    return math.acos(max(-1.0, min(1.0, R[2, 2])))

def heading(q):
    """yaw of body +X projected on the horizontal plane. Invariant under a roll about body X."""
    R = q_to_R(q)
    return math.atan2(R[1, 0], R[0, 0])

def wrap(a):
    return (a + math.pi) % (2*math.pi) - math.pi

# ---------------------------------------------------------------- configuration
def default_nozzles():
    """L1: HYPOTHETICAL 10-nozzle layout. name, group, position (m, body), EXHAUST dir, Fmax (N)."""
    rows = []
    for i, y in enumerate([-0.7, -0.3, 0.3, 0.7]):
        rows.append((f"rear{i}", "rear", (-2.1, y, 0.0), (-1, 0, 0), 20000.0))     # plume rearward -> force forward
    for name, x, y in [("cornerFL", 1.7, 0.9), ("cornerFR", 1.7, -0.9),
                       ("cornerRL", -1.7, 0.9), ("cornerRR", -1.7, -0.9)]:
        rows.append((name, "corner", (x, y, -0.20), (0, 0, -1), 8000.0))              # plume to floor -> force up
    rows.append(("yawL", "yaw", (1.9, 0.6, 0.0), (0, 1, 0), 3000.0))                  # plume left -> force right
    rows.append(("yawR", "yaw", (1.9, -0.6, 0.0), (0, -1, 0), 3000.0))                # plume right -> force left
    return rows

@dataclass
class VehicleConfig:
    m_total0: float = 1700.0        # kg, including gas at start (assumption)
    gas0: float = 250.0             # kg, reported at the event (assumed usable)
    dims: tuple = (4.3, 2.0, 1.3)   # L, W, H  (m)  illustrative
    inertia: tuple = None           # None -> uniform box
    nozzles: list = field(default_factory=default_nozzles)
    valve_tau: float = 0.03         # s, assumed
    isp: float = 70.0               # s, assumed cold-gas Isp
    gas_depletion: bool = True      # mass and inertia scale with remaining gas
    wheel_radius: float = 0.33
    wheel_xy: tuple = (1.35, 0.9)   # |x|, |y| of wheel centres
    wheel_z: float = -0.55          # wheel centre below CoM; gives ~0.23 m of landing-gear travel before the chassis touches (assumption)
    k_wheel: float = 1.0e5
    c_wheel: float = 6.0e3
    k_chassis: float = 5.0e5
    c_chassis: float = 2.0e4
    mu_wheel_lat: float = 0.9
    mu_wheel_roll: float = 0.03
    mu_chassis: float = 0.6
    v_eps: float = 0.1
    pen_tol: float = 1e-4           # m: contact detection tolerance (numerical)
    crash_pen: float = 0.02         # m: chassis penetration that counts as a crash
    dt: float = 0.005

    def box_inertia(self):
        L, W, H = self.dims; m = self.m_total0
        return (m/12*(W*W + H*H), m/12*(L*L + H*H), m/12*(L*L + W*W))

    def nozzle_table(self):
        import pandas as pd
        return pd.DataFrame([dict(name=n, group=g, x=p[0], y=p[1], z=p[2], ex=e[0], ey=e[1], ez=e[2], fmax_N=f)
                             for n, g, p, e, f in self.nozzles])

class Sim3D:
    def __init__(self, cfg: VehicleConfig = None):
        self.cfg = cfg or VehicleConfig()
        self.configure()

    def configure(self):
        c = self.cfg
        self.m0 = c.m_total0
        self.I0 = np.array(c.inertia if c.inertia is not None else c.box_inertia(), float)
        pos = np.array([n[2] for n in c.nozzles], float)
        ex = np.array([n[3] for n in c.nozzles], float)
        ex /= np.linalg.norm(ex, axis=1, keepdims=True)
        self.nz_pos = pos
        self.nz_exhaust = ex
        self.nz_fdir = -ex                                   # reaction direction on the vehicle
        self.nz_tmat = np.cross(pos, self.nz_fdir)           # torque per newton
        self.fmax = np.array([n[4] for n in c.nozzles], float)
        self.nn = len(c.nozzles)
        L, W, H = c.dims
        corners = np.array([[sx*L/2, sy*W/2, sz*H/2] for sx in (1, -1) for sy in (1, -1) for sz in (1, -1)])
        wx, wy = c.wheel_xy
        wheels = np.array([[wx, wy, c.wheel_z], [wx, -wy, c.wheel_z], [-wx, wy, c.wheel_z], [-wx, -wy, c.wheel_z]])
        self.cp_body = np.vstack([wheels, corners])
        self.cp_rad = np.concatenate([np.full(4, c.wheel_radius), np.zeros(8)])
        self.cp_k = np.concatenate([np.full(4, c.k_wheel), np.full(8, c.k_chassis)])
        self.cp_c = np.concatenate([np.full(4, c.c_wheel), np.full(8, c.c_chassis)])
        self.valve_alpha = 1 - math.exp(-c.dt / c.valve_tau)
        self.reset()

    # ------------------------------------------------------------------ state
    def reset(self, p=(0, 0, 5.0), v=(0, 0, 0), q=(1, 0, 0, 0), w=(0, 0, 0), gas=None, f=None):
        self.p = np.array(p, float); self.v = np.array(v, float)
        self.q = qnorm(np.array(q, float)); self.w = np.array(w, float)
        self.gas = self.cfg.gas0 if gas is None else float(gas)
        self.f = np.zeros(self.nn) if f is None else np.array(f, float)
        self.t = 0.0
        self.last = dict(wheel_n=0, chassis_n=0, max_pen_chassis=0.0, fn_total=0.0)
        self.gas_used = 0.0
        self.impulse = 0.0

    @property
    def mass(self):
        return self.m0 - (self.cfg.gas0 - self.gas) if self.cfg.gas_depletion else self.m0

    @property
    def inertia(self):
        return self.I0 * (self.mass / self.m0)

    def wrench_body(self, f=None):
        f = self.f if f is None else f
        return f @ self.nz_fdir, f @ self.nz_tmat

    # ---------------------------------------------------------------- contact
    def contact(self, R):
        c = self.cfg
        rw = self.cp_body @ R.T
        pen = self.cp_rad - (self.p[2] + rw[:, 2])
        act = pen > c.pen_tol
        F = np.zeros(3); T = np.zeros(3)
        self.last = dict(wheel_n=0, chassis_n=0, max_pen_chassis=0.0, fn_total=0.0)
        if not act.any():
            return F, T
        ww = R @ self.w
        for i in np.nonzero(act)[0]:
            rc = rw[i].copy(); rc[2] -= self.cp_rad[i]            # contact point offset from CoM
            vp = self.v + np.cross(ww, rc)
            fn = self.cp_k[i]*pen[i] - self.cp_c[i]*vp[2]
            if fn <= 0.0:
                continue
            vx, vy = vp[0], vp[1]
            if i < 4:
                hx, hy = R[0, 0], R[1, 0]; n = math.hypot(hx, hy)
                fx, fy = (hx/n, hy/n) if n > 1e-3 else (1.0, 0.0)
                vl = vx*fx + vy*fy; vs = -vx*fy + vy*fx
                fl = -fn*c.mu_wheel_roll*vl/math.sqrt(vl*vl + c.v_eps**2)
                fs = -fn*c.mu_wheel_lat*vs/math.sqrt(vs*vs + c.v_eps**2)
                ft_x = fl*fx - fs*fy; ft_y = fl*fy + fs*fx
                self.last['wheel_n'] += 1
            else:
                vt = math.sqrt(vx*vx + vy*vy + c.v_eps**2)
                ft_x = -fn*c.mu_chassis*vx/vt; ft_y = -fn*c.mu_chassis*vy/vt
                self.last['chassis_n'] += 1
                self.last['max_pen_chassis'] = max(self.last['max_pen_chassis'], pen[i])
            Fi = np.array([ft_x, ft_y, fn])
            F += Fi; T += np.cross(rc, Fi)
            self.last['fn_total'] += fn
        return F, T   # forces and torques in the WORLD frame

    # ------------------------------------------------------------------- step
    def substep(self, u):
        c = self.cfg; dt = c.dt
        target = np.clip(u, 0.0, 1.0) * self.fmax
        if self.gas <= 0.0:
            target = target*0.0
        self.f += (target - self.f) * self.valve_alpha
        if self.gas <= 0.0:
            self.f *= 0.0
        m = self.mass; I = self.inertia
        Fb, Tb = self.wrench_body()
        R = q_to_R(self.q)
        Fc, Tc_w = self.contact(R)
        tau = Tb + R.T @ Tc_w
        wd = (tau - np.cross(self.w, I*self.w)) / I
        w_mid = self.w + 0.5*dt*wd
        q_half = qmul(self.q, q_expmap(w_mid*dt*0.5))
        R_mid = q_to_R(q_half)
        a = (R_mid @ Fb + Fc) / m + np.array([0, 0, -G0])
        self.p = self.p + self.v*dt + 0.5*a*dt*dt
        self.v = self.v + a*dt
        wd_mid = (tau - np.cross(w_mid, I*w_mid)) / I
        self.w = self.w + dt*wd_mid
        self.q = qnorm(qmul(self.q, q_expmap(w_mid*dt)))
        fs = float(self.f.sum())
        used = fs/(c.isp*G0)*dt            # gas is always counted; mass/inertia change only if gas_depletion
        self.gas = max(0.0, self.gas - used); self.gas_used += used
        self.impulse += fs*dt
        self.t += dt

    def step(self, u, n_sub=4):
        for _ in range(n_sub):
            self.substep(u)

    # ---------------------------------------------------------------- queries
    def energy(self):
        I = self.inertia; m = self.mass
        return 0.5*m*float(self.v @ self.v) + 0.5*float(np.sum(I*self.w*self.w)) + m*G0*self.p[2]

    def ang_momentum_world(self):
        I = self.inertia
        return q_to_R(self.q) @ (I*self.w)

    def elastic_energy(self):
        R = q_to_R(self.q)
        pen = self.cp_rad - (self.p[2] + (self.cp_body @ R.T)[:, 2])
        a = pen > self.cfg.pen_tol
        return float(np.sum(0.5*self.cp_k[a]*pen[a]**2))

    def lowest_point_z(self):
        R = q_to_R(self.q)
        return float(np.min(self.p[2] + (self.cp_body @ R.T)[:, 2] - self.cp_rad))

In [ ]:
%%writefile env3d.py
"""
Gymnasium environments for the Roadster-inspired 3D tasks (shared physics: sim3d.Sim3D).

Tasks: landing, recover90, recover180, roll360 (aerial), kickflip (ground launch, +360 deg body-X roll over an obstacle lineup),
       ramp_exit (A1: inverted ramp-exit conditions, about +180 deg body-X roll recovery; added in ramp.py)
Reward: dense tracking costs (always <= 0), potential-based shaping with the SAME gamma PPO uses (terminal potential = 0),
        large sparse terminal terms. Success terms are gated by genuine signed rotation progress.
"""
import math
import numpy as np
import gymnasium as gym
from gymnasium import spaces
from sim3d import *

REST_Z = 0.8383          # CoM height when resting on the wheels (from the drop test)
OBSTACLES = [(11.0, 20.0, 1.6), (13.5, 17.0, 3.5)]   # (x0, x1, height m): car lineup + Semi-like block (ILLUSTRATIVE, placed around the flight apex)
OBST_HALF_WIDTH = 2.5

def roll_profile(t, t0, T, total):
    """bang-bang (triangular velocity) roll profile about body X: theta, theta_dot, theta_ddot"""
    x = t - t0
    if total == 0.0 or x <= 0.0:
        return 0.0, 0.0, 0.0
    if x >= T:
        return total, 0.0, 0.0
    a = 4*total/T**2
    if x < T/2:
        return 0.5*a*x*x, a*x, a
    y = T - x
    return total - 0.5*a*y*y, a*y, -a

def build_kick_reference(v_l=7.0, a_b=7.5, T_roll=1.4, a_land=7.0, v_td=-1.0, dt=0.002):
    """Piecewise-constant-acceleration height reference: launch burn, ballistic, landing burn. Returns arrays (t, z, vz)."""
    t_b = v_l/a_b
    ts, zs, vs = [0.0], [0.0], [0.0]
    t = 0.0; z = 0.0; v = 0.0; phase = 0
    while t < 8.0:
        if phase == 0:
            a = a_b
            if v >= v_l: phase = 1
        if phase == 1:
            a = -G0
            if v < 0 and (v*v - v_td*v_td)/(2*a_land) >= z and t > t_b + T_roll*0.8: phase = 2
        if phase == 2:
            a = a_land
        v += a*dt; z += v*dt; t += dt
        if phase == 2 and (z <= 0.0 or v >= v_td):
            z = max(z, 0.0); ts.append(t); zs.append(z); vs.append(v); break
        ts.append(t); zs.append(z); vs.append(v)
    return np.array(ts), np.array(zs) + REST_Z, np.array(vs)

KICK_REF = build_kick_reference()
import os as _os
_p = _os.path.join(_os.path.dirname(_os.path.abspath(__file__)), 'kick_ref.npz')
if _os.path.exists(_p):                      # height reference taken from the physically simulated scripted run (feasible by construction)
    _d = np.load(_p); KICK_REF = (_d['t'], _d['z'], _d['vz'])

TASKS = dict(
    landing=dict(T_max=6.0, roll_total=0.0),
    recover90=dict(T_max=7.0, roll_total=0.0),
    recover180=dict(T_max=7.0, roll_total=0.0),
    roll360=dict(T_max=7.0, roll_total=2*math.pi),
    kickflip=dict(T_max=5.0, roll_total=2*math.pi),
)

class RoadsterEnv(gym.Env):
    metadata = {"render_modes": []}
    DT = 0.02

    def __init__(self, task="kickflip", cfg=None, gamma=0.99, seed=None, T_roll=1.4, t_roll0=None, roll_sign=1.0,
                 randomize_cfg=0.0, record=False, cfg_factory=None):
        super().__init__()
        self.task = task; self.spec = dict(TASKS.get(task, {}))
        self.cfg = cfg or VehicleConfig()
        self.sim = Sim3D(self.cfg)
        self.gamma = gamma; self.T_roll = T_roll; self.roll_sign = roll_sign
        self.randomize_cfg = randomize_cfg; self.record = record
        self.t_roll0_fixed = t_roll0
        self.rng = np.random.default_rng(seed)
        self.n_nz = self.sim.nn
        self.obs_dim = 54 if self.n_nz == 10 else 44 + self.n_nz
        self.observation_space = spaces.Box(-np.inf, np.inf, (self.obs_dim,), np.float32)
        self.action_space = spaces.Box(-1.0, 1.0, (self.n_nz,), np.float32)
        self.base = dict(I0=self.sim.I0.copy(), m0=self.sim.m0, fmax=self.sim.fmax.copy())

    # ------------------------------------------------------------ reference
    def ref_att(self, t):
        th, thd, thdd = roll_profile(t, self.t_roll0, self.T_roll, self.roll_total)
        q = qmul(self.q_ref0, q_axis_angle([1, 0, 0], th))
        return q, np.array([thd, 0, 0]), np.array([thdd, 0, 0]), th

    def ref_pos(self, t):
        if self.task == "kickflip":
            T, Z, V = KICK_REF
            z = np.interp(t, T, Z); vz = np.interp(t, T, V)
            return np.array([self.x0 + self.vx0*t, 0.0, z]), np.array([self.vx0, 0.0, vz])
        return None, None

    # ------------------------------------------------------------ reset
    def reset(self, seed=None, options=None):
        if seed is not None:
            self.rng = np.random.default_rng(seed)
        rng = self.rng; task = self.task
        # optional domain randomisation of mass / inertia / thrust (sensitivity studies)
        k = self.randomize_cfg
        s = self.sim
        s.m0 = self.base['m0']*(1 + k*rng.uniform(-1, 1)); s.I0 = self.base['I0']*(1 + k*rng.uniform(-1, 1, 3))
        s.fmax = self.base['fmax']*(1 + k*rng.uniform(-1, 1))
        self.roll_total = self.spec['roll_total']*self.roll_sign
        yaw0 = rng.uniform(-0.05, 0.05)
        q_up = q_from_euler_zyx(yaw0, 0, 0)
        self.t_roll0 = 0.0; self.x0 = 0.0; self.vx0 = 0.0
        if task == "landing":
            p = (rng.uniform(-2, 2), rng.uniform(-2, 2), rng.uniform(3.0, 8.0))
            v = (rng.uniform(-3, 3), rng.uniform(-1.5, 1.5), rng.uniform(-6, 0))
            q = qmul(q_up, q_expmap(rng.normal(0, 0.08, 3))); w = rng.normal(0, 0.15, 3)
            self.q_ref0 = q_up; gas = None
        elif task in ("recover90", "recover180"):
            off = (math.pi/2 if task == "recover90" else math.pi)*rng.choice([-1, 1])
            if task == "recover180": off = math.pi*rng.uniform(0.9, 1.0)*rng.choice([-1, 1])
            hlo, hhi = (10, 14) if task == 'recover90' else (16, 22)      # an inverted car needs the altitude to roll over AND brake
            p = (rng.uniform(-2, 2), rng.uniform(-2, 2), rng.uniform(hlo, hhi))
            v = (rng.uniform(-3, 3), rng.uniform(-1, 1), rng.uniform(-2, 3))
            q = qmul(qmul(q_up, q_axis_angle([1, 0, 0], off)), q_expmap(rng.normal(0, 0.05, 3)))
            w = np.array([rng.normal(0, 0.3), rng.normal(0, 0.15), rng.normal(0, 0.15)])
            self.q_ref0 = q_up
        elif task == "roll360":
            p = (rng.uniform(-2, 2), rng.uniform(-1, 1), rng.uniform(11, 15))
            v = (rng.uniform(-2, 2), rng.uniform(-0.5, 0.5), rng.uniform(2, 6))
            q = qmul(q_up, q_expmap(rng.normal(0, 0.03, 3))); w = rng.normal(0, 0.05, 3)
            self.q_ref0 = q_up; self.t_roll0 = self.t_roll0_fixed if self.t_roll0_fixed is not None else 0.1
        elif task == "kickflip":
            self.vx0 = rng.uniform(8.0, 10.0); self.x0 = rng.uniform(-1.0, 1.0)
            p = (self.x0, rng.uniform(-0.15, 0.15), REST_Z)
            v = (self.vx0, rng.normal(0, 0.1), 0.0)
            q = q_up; w = np.zeros(3); self.q_ref0 = q_up
            self.t_roll0 = self.t_roll0_fixed if self.t_roll0_fixed is not None else 0.95 + rng.uniform(-0.05, 0.05)
        else:
            raise ValueError(task)
        self._extra_reset(rng)  # hook for ramp tasks
        if task in ("landing", "recover90", "recover180", "roll360", "kickflip"):
            s.reset(p=p, v=v, q=q, w=w)
        self.sim.t = 0.0
        self.phi = 0.0; self._twist_prev = 0.0
        self.airborne = False; self.touch_t = None; self.touch = None
        self.crashed = False; self.collided = False; self.landed = False
        self.min_clear = 99.0
        self.step_n = 0; self.u_prev = np.zeros(self.n_nz); self.hist = [] if self.record else None
        self.max_phi = 0.0; self.ret = 0.0; self.maxz = float(self.sim.p[2])
        self.prev_pot = self._potential(self._state_feats())
        return self._obs(), {}

    def _extra_reset(self, rng):
        pass

    # ------------------------------------------------------------ features
    def _update_progress(self):
        qr = qmul(qconj(self.q_ref0), self.sim.q)
        if qr[0]**2 + qr[1]**2 > 1e-4:
            tw = 2*math.atan2(qr[1], qr[0])
            self.phi += wrap(tw - self._twist_prev); self._twist_prev = tw
        self.max_phi = max(self.max_phi, self.phi*self.roll_sign)

    def _state_feats(self):
        s = self.sim; t = s.t
        q_ref, w_ref, a_ref, th_ref = self.ref_att(t)
        e_att = geodesic(s.q, q_ref)
        return dict(q_ref=q_ref, w_ref=w_ref, th_ref=th_ref, e_att=e_att)

    def _potential(self, f):
        """>= 0, bounded, evaluated on non-terminal states; terminal potential is defined as 0 in step()."""
        goal = abs(self.roll_total)
        prog = 0.0
        if goal > 0:
            prog = min(1.0, max(0.0, self.phi*self.roll_sign/goal))
        att = math.exp(-(f['e_att']/0.5)**2)
        return 12.0*prog + 6.0*att

    def _clearance(self):
        """vertical clearance (m) of the lowest car point over any obstacle it is above; large if none"""
        s = self.sim; R = q_to_R(s.q)
        pts = s.p + s.cp_body @ R.T
        pts[:4, 2] -= s.cp_rad[:4]
        best = 99.0; hit = False
        for (x0, x1, h) in self.obstacles():
            m = (pts[:, 0] > x0) & (pts[:, 0] < x1) & (np.abs(pts[:, 1]) < OBST_HALF_WIDTH)
            if m.any():
                c = float(np.min(pts[m, 2]) - h)
                best = min(best, c)
                if c < 0: hit = True
        return best, hit

    def obstacles(self):
        return OBSTACLES if self.task == "kickflip" else []

    def _obs(self):
        s = self.sim; R = q_to_R(s.q); t = s.t
        f = self._state_feats()
        e_vec = q_err_vec(s.q, f['q_ref'])
        p_ref, v_ref = self.ref_pos(t)
        if p_ref is None:
            ep = np.zeros(3); ev = np.zeros(3)
        else:
            ep = p_ref - s.p; ev = v_ref - s.v
        clear, _ = self._clearance() if self.obstacles() else (99.0, False)
        nxt = [x0 for (x0, x1, h) in self.obstacles() if x1 > s.p[0]]
        dx = (min(nxt) - s.p[0]) if nxt else 40.0
        th_ref = f['th_ref']
        o = np.concatenate([
            R.flatten(), e_vec/math.pi, s.w/5.0, f['w_ref']/5.0, (R.T @ s.v)/10.0, s.v/10.0, [s.p[2]/10.0],
            ep/5.0, ev/5.0, (R.T @ ep)/5.0,
            [self.phi*self.roll_sign/(2*math.pi), th_ref*self.roll_sign/(2*math.pi)] if True else [],
            [s.t/self.spec['T_max']], [s.gas/s.cfg.gas0], s.f/s.fmax,
            [s.last['wheel_n']/4.0, float(s.last['chassis_n'] > 0)],
            [min(clear, 10.0)/5.0, np.clip(dx, -10, 40)/20.0],
            [float(self.airborne), float(self.landed)]])
        return o.astype(np.float32)

    # ------------------------------------------------------------ step
    def step(self, action):
        s = self.sim
        a = np.clip(np.asarray(action, float), -1, 1)
        u = 0.5*(a + 1.0)
        s.step(u, 4)
        self._update_progress()
        self.step_n += 1
        t = s.t
        lowest = s.lowest_point_z()
        wheel_n = s.last['wheel_n']; chassis_n = s.last['chassis_n']
        if lowest > 0.15: self.airborne = True
        self.maxz = max(self.maxz, s.p[2])
        clear, hit = self._clearance() if self.obstacles() else (99.0, False)
        self.min_clear = min(self.min_clear, clear)
        if hit: self.collided = True
        if s.last['max_pen_chassis'] > s.cfg.crash_pen: self.crashed = True
        if self.airborne and wheel_n > 0 and self.touch_t is None:
            R = q_to_R(s.q)
            self.touch_t = t
            self.touch = dict(vz=float(s.v[2]), vh=float(np.hypot(s.v[0], s.v[1])), w=s.w.copy(), tilt=tilt_angle(s.q),
                              phi=self.phi, v_body=(R.T @ s.v).copy(), p=s.p.copy())
        f = self._state_feats()
        # ---------------- costs
        e_att = f['e_att']; e_w = float(np.linalg.norm(s.w - f['w_ref']))
        p_ref, v_ref = self.ref_pos(t)
        cost = 1.0*(e_att/0.7)**2 + 0.15*(e_w/3.0)**2
        if p_ref is not None:
            ep = p_ref - s.p
            cost += 0.25*((np.hypot(ep[0], ep[1])/4.0)**2 + (ep[2]/2.5)**2)
        phase_done = (abs(self.roll_total) == 0) or (self.phi*self.roll_sign > abs(self.roll_total) - 0.5)
        if phase_done:    # descent-speed shaping: allowed downward speed shrinks with height
            h = max(lowest, 0.0)
            v_allow = 1.0 + math.sqrt(2*6.0*h)
            cost += 0.3*(max(0.0, -s.v[2] - v_allow)/3.0)**2
        cost += 0.02*float(np.mean(u**2)) + 0.02*float(np.mean((u - self.u_prev)**2))*10
        self.u_prev = u
        r = -0.05*min(cost, 4.0)
        # ---------------- termination
        terminated = False; truncated = False; success = False; reason = ""
        if self.crashed:
            terminated = True; reason = "chassis_contact"; r -= 50.0
        elif self.collided:
            terminated = True; reason = "obstacle_collision"; r -= 50.0
        elif self.touch_t is not None:
            maneuver_done = (abs(self.roll_total) == 0) or (self.phi*self.roll_sign > abs(self.roll_total) - math.radians(30))
            if not maneuver_done:
                terminated = True; reason = "touchdown_before_maneuver_complete"; r -= 40.0
            elif t - self.touch_t >= 0.6:
                terminated = True; self.landed = True
                success, info_s = self._success()
                reason = "landed"; r += 60.0 if success else -10.0
        elif s.p[2] > 40 or abs(s.p[0]) > 80 or abs(s.p[1]) > 40:
            terminated = True; reason = "out_of_bounds"; r -= 50.0
        if not terminated and t >= self.spec['T_max'] - 1e-9:
            truncated = True; reason = "timeout"
        # ---------------- potential-based shaping: gamma*Phi(s') - Phi(s), terminal potential = 0
        pot = 0.0 if terminated else self._potential(f)
        r += self.gamma*pot - self.prev_pot
        self.prev_pot = pot
        self.ret += r
        info = {}
        if self.record:
            self.hist.append(dict(t=t, p=s.p.copy(), v=s.v.copy(), q=s.q.copy(), w=s.w.copy(), f=s.f.copy(), u=u.copy(),
                                  phi=self.phi, th_ref=f['th_ref'], e_att=e_att, gas=s.gas, wheel_n=wheel_n, chassis_n=chassis_n,
                                  q_ref=f['q_ref'].copy(), clear=clear, r=r))
        if terminated or truncated:
            info = self._metrics(success, reason)
            info['TimeLimit.truncated'] = bool(truncated)
        return self._obs(), float(r), terminated, truncated, info

    # ------------------------------------------------------------ evaluation
    def _success(self):
        s = self.sim; tch = self.touch
        f = self._state_feats()
        tilt = tilt_angle(s.q); wn = float(np.linalg.norm(s.w))
        vz_ok = -tch['vz'] <= 1.5
        slip_ok = abs(tch['v_body'][1]) <= 1.5
        head_err = abs(wrap(heading(s.q) - heading(self.q_ref0)))
        roll_ok = (abs(self.roll_total) == 0) or abs(self.phi*self.roll_sign - abs(self.roll_total)) < math.radians(30)
        checks = dict(tilt_ok=tilt < math.radians(10), rate_ok=wn < 0.3, vz_ok=vz_ok, slip_ok=slip_ok,
                      heading_ok=head_err < math.radians(20), roll_ok=roll_ok, wheels_ok=s.last['wheel_n'] >= 3,
                      no_chassis=not self.crashed, no_collision=not self.collided, gas_ok=s.gas > 0)
        return all(checks.values()), checks

    def _metrics(self, success, reason):
        s = self.sim
        ok, checks = (self._success() if self.touch is not None else (False, {}))
        tch = self.touch or {}
        m = dict(success=bool(success), reason=reason, task=self.task, t_end=s.t,
                 maneuver_progress_deg=math.degrees(self.phi*self.roll_sign), max_progress_deg=math.degrees(self.max_phi),
                 maneuver_complete=bool((abs(self.roll_total) == 0) or self.max_phi > abs(self.roll_total) - math.radians(30)),
                 final_tilt_deg=math.degrees(tilt_angle(s.q)), final_rate=float(np.linalg.norm(s.w)),
                 touch_vz=tch.get('vz', float('nan')), touch_vh=tch.get('vh', float('nan')),
                 touch_rates=(tch['w'].tolist() if 'w' in tch else None), touch_tilt_deg=math.degrees(tch['tilt']) if 'tilt' in tch else float('nan'),
                 wheel_contacts=s.last['wheel_n'], chassis_contact=bool(self.crashed), obstacle_collision=bool(self.collided),
                 min_obstacle_clearance=self.min_clear, gas_left=s.gas, peak_height=self.maxz - REST_Z, ret=self.ret,
                 heading_err_deg=math.degrees(abs(wrap(heading(s.q) - heading(self.q_ref0)))), checks=checks)
        p_ref, _ = self.ref_pos(s.t)
        m['final_xy'] = s.p[:2].tolist()
        return m

In [ ]:
%%writefile ramp.py
"""
Task A1 (Roadster Ramp Recovery, airborne part): start from ramp-EXIT conditions, recover about +180 deg of body-X roll, land on the wheels.
The ramp's contribution is represented ONLY through the initial state (about -170 deg of pitch, i.e. nose-up past vertical, high altitude,
some pitch rate). The full ramp contact stage (A2) is not part of this environment.
Heading: a roll about body X leaves the horizontal heading of body +X unchanged, so after ramp pitch (about 170 deg) the car faces
back toward where it came from; the target heading is that post-ramp heading (heading(q_exit)), not the pre-ramp heading.
"""
import math
import numpy as np
from sim3d import *
from env3d import *

TASKS['ramp_exit'] = dict(T_max=6.0, roll_total=math.pi)

class RampExitEnv(RoadsterEnv):
    def __init__(self, **kw):
        kw.setdefault('T_roll', 1.2)
        super().__init__(task="ramp_exit", **kw)

    def reset(self, seed=None, options=None):
        if seed is not None: self.rng = np.random.default_rng(seed)
        rng = self.rng; s = self.sim
        k = self.randomize_cfg
        s.m0 = self.base['m0']*(1 + k*rng.uniform(-1, 1)); s.I0 = self.base['I0']*(1 + k*rng.uniform(-1, 1, 3))
        s.fmax = self.base['fmax']*(1 + k*rng.uniform(-1, 1))
        self.roll_total = self.spec['roll_total']*self.roll_sign
        yaw0 = rng.uniform(-0.1, 0.1)
        pitch = -math.radians(170 + rng.uniform(-8, 8))                  # nose-up past vertical (ramp-induced), reported ~170 deg
        q = qmul(qmul(q_axis_angle([0, 0, 1], yaw0), q_axis_angle([0, 1, 0], pitch)), q_expmap(rng.normal(0, 0.03, 3)))
        self.q_exit = q; self.q_ref0 = q
        self.t_roll0 = 0.15
        q_end = qmul(q, q_axis_angle([1, 0, 0], self.roll_total))
        q_lvl = q_from_euler_zyx(heading(q_end), 0, 0)
        self.q_corr = qmul(q_lvl, qconj(q_end)); 
        if self.q_corr[0] < 0: self.q_corr = -self.q_corr
        self.x0 = 0.0; self.vx0 = 0.0
        p = (rng.uniform(-1, 1), rng.uniform(-1, 1), rng.uniform(18.0, 21.0))
        v = (rng.uniform(-3, 3), rng.uniform(-1, 1), rng.uniform(-3, 5))
        w = np.array([rng.normal(0, 0.15), rng.uniform(-0.8, 0.0), rng.normal(0, 0.15)])   # residual ramp pitch rate (nose-up sense)
        s.reset(p=p, v=v, q=q, w=w)
        self.sim.t = 0.0
        self.phi = 0.0; self._twist_prev = 0.0
        self.airborne = True; self.touch_t = None; self.touch = None
        self.crashed = False; self.collided = False; self.landed = False; self.min_clear = 99.0
        self.step_n = 0; self.u_prev = np.zeros(self.n_nz); self.hist = [] if self.record else None
        self.max_phi = 0.0; self.ret = 0.0; self.maxz = float(self.sim.p[2])
        self.prev_pot = self._potential(self._state_feats())
        return self._obs(), {}

    def ref_att(self, t):
        th, thd, thdd = roll_profile(t, self.t_roll0, self.T_roll, self.roll_total)
        frac = th/self.roll_total if self.roll_total != 0 else 0.0
        qc = q_expmap(frac*self._qlog(self.q_corr))
        q = qmul(qc, qmul(self.q_exit, q_axis_angle([1, 0, 0], th)))
        return q, np.array([thd, 0, 0]), np.array([thdd, 0, 0]), th

    @staticmethod
    def _qlog(q):
        s = math.sqrt(q[1]**2 + q[2]**2 + q[3]**2)
        if s < 1e-12: return np.zeros(3)
        return q[1:4]*(2*math.atan2(s, q[0])/s)

    def obstacles(self): return []

    def _success(self):
        ok, checks = super()._success()
        return ok, checks

In [ ]:
%%writefile scripted.py
"""
Classical (NON-learned) reference controller. Used ONLY to (1) prove the nozzle layout has the authority to perform the maneuver,
(2) produce a phase-by-phase feasibility record, (3) optionally provide demonstrations. Results from this file are always
labelled 'scripted', never 'PPO'.
"""
import math
import numpy as np
from scipy.optimize import lsq_linear
from sim3d import *
from env3d import *

class ScriptedController:
    def __init__(self, env, kp=60.0, kd=14.0, a_burn=7.5, a_land=7.0, v_td=-1.0, v_launch=6.25, margin=1.1, lat_ctrl=None):
        self.env = env; self.kp = kp; self.kd = kd; self.a_burn = a_burn; self.a_land = a_land; self.v_td = v_td
        self.v_launch = v_launch; self.margin = margin
        self.lat_ctrl = (env.task != 'kickflip') if lat_ctrl is None else lat_ctrl
        s = env.sim
        self.A = np.vstack([s.nz_fdir.T, s.nz_tmat.T])      # 6 x n : [Fx Fy Fz tx ty tz]
        self.reset()

    def reset(self):
        self.landing_burn = False; self.phase = "launch"; self.launched = False
        self.skip_roll = getattr(self, 'skip_roll', False); self.hover_alt = getattr(self, 'hover_alt', None)

    def _alloc(self, wrench, weights):
        s = self.env.sim
        sc = np.array([1e4, 1e4, 1e4, 1e4, 1e4, 1e4])
        W = np.asarray(weights)/sc
        A = self.A*W[:, None]; b = np.asarray(wrench)*W
        reg = 1e-4*np.eye(s.nn)/1e4*0   # no hidden regularisation; tiny ridge below
        A2 = np.vstack([A, 1e-3*np.eye(s.nn)/s.fmax[None, :]*1.0]); b2 = np.concatenate([b, np.zeros(s.nn)])
        r = lsq_linear(A2, b2, bounds=(np.zeros(s.nn), s.fmax), method="bvls")
        return r.x

    def act(self):
        env = self.env; s = env.sim; t = s.t; m = s.mass; I = s.inertia
        R = q_to_R(s.q)
        q_ref, w_ref, a_ref, th = env.ref_att(t)
        if getattr(self, 'skip_roll', False):                   # exploit test: ignore the roll reference
            q_ref, w_ref, a_ref = env.q_ref0, np.zeros(3), np.zeros(3)
        rolling0 = abs(env.roll_total) > 0 and (env.phi*env.roll_sign < abs(env.roll_total) - 0.15)
        if self.lat_ctrl and not rolling0 and not (env.touch_t is not None):
            vby = float((R.T @ s.v)[1]); dlt = float(np.clip(0.10*vby, -0.25, 0.25))      # tilt the thrust vector to null body-lateral velocity
            q_ref = qmul(q_ref, q_axis_angle([1, 0, 0], dlt))
        e = q_err_vec(s.q, q_ref)
        tau = I*(self.kp*e + self.kd*(w_ref - s.w) + a_ref) + np.cross(s.w, I*s.w)
        lowest = s.lowest_point_z()
        Fz = 0.0; wz = 0.0
        task = env.task
        roll_total = abs(env.roll_total)
        rolling = roll_total > 0 and (env.phi*env.roll_sign < roll_total - 0.15) and not getattr(self, 'skip_roll', False)
        done_roll = (roll_total == 0) or not rolling
        if s.last['wheel_n'] > 0 and env.airborne and env.touch_t is not None:
            self.phase = "touchdown"; return -np.ones(s.nn)
        if task == "kickflip" and not self.launched:
            self.phase = "launch"
            if s.v[2] >= self.v_launch or t >= env.t_roll0 + 0.3:
                self.launched = True
            else:
                Fz = m*(G0 + self.a_burn)/max(R[2, 2], 0.5); wz = 1.0
        if task != "kickflip" or self.launched:
            if task == "kickflip" and rolling:
                self.phase = "roll"; wz = 0.0
            elif task == "roll360" and rolling:
                self.phase = "roll"; wz = 0.0
            else:
                self.phase = "recover/descent"
                h = max(lowest, 0.05); vz = s.v[2]
                if vz < 0:
                    a_req = (vz*vz - self.v_td**2)/(2*h)
                    if (not self.landing_burn) and a_req >= self.a_land*0.9:
                        self.landing_burn = True
                    if self.landing_burn:
                        self.phase = "landing burn"
                        if vz < self.v_td:
                            a_des = max(a_req*self.margin, 0.0)
                        else:
                            a_des = 3.0*(self.v_td - vz)        # already slow enough: hold the touchdown speed
                        Fz = m*(G0 + a_des)/max(R[2, 2], 0.5); wz = 1.0
        if getattr(self, 'hover_alt', None) is not None and self.launched:     # exploit test: hover instead of landing
            Fz = m*(G0 + 2.0*(self.hover_alt - s.p[2]) - 1.5*s.v[2])/max(R[2, 2], 0.5); wz = 1.0; self.phase = 'hover'
        wrench = [0, 0, Fz, tau[0], tau[1], tau[2]]
        weights = [0, 0, wz*1.0, 3.0, 3.0, 3.0]
        x = self._alloc(wrench, weights)
        return 2*np.clip(x/s.fmax, 0, 1) - 1

def run_scripted(task="kickflip", seed=0, record=True, **kw):
    env = RoadsterEnv(task, seed=seed, record=record, **kw)
    ctl = ScriptedController(env)
    o, _ = env.reset(seed=seed); ctl.reset()
    phases = []; info = {}
    while True:
        a = ctl.act(); phases.append(ctl.phase)
        o, r, te, tr, info = env.step(a)
        if te or tr: break
    return env, info, phases

In [ ]:
%%writefile ppo_np.py
"""
Minimal PPO (clip objective, GAE, Gaussian policy, running observation normalisation) in pure NumPy with manual back-propagation.
Used because it is fast on a single CPU core and has no framework dependency. Same code runs in Colab.
"""
import time, json, math, os
import numpy as np

class RunningNorm:
    def __init__(self, dim):
        self.n = 1e-4; self.mean = np.zeros(dim); self.var = np.ones(dim)
    def update(self, x):
        bm = x.mean(0); bv = x.var(0); bn = x.shape[0]
        d = bm - self.mean; tot = self.n + bn
        self.mean = self.mean + d*bn/tot
        self.var = (self.var*self.n + bv*bn + d*d*self.n*bn/tot)/tot; self.n = tot
    def __call__(self, x):
        return np.clip((x - self.mean)/np.sqrt(self.var + 1e-8), -8, 8)

def _init(rng, i, o, scale=1.0):
    return rng.normal(0, 1, (i, o))*scale/math.sqrt(i)

class MLP:
    """obs -> h1 -> h2 -> out, tanh hidden units"""
    def __init__(self, rng, i, o, h=96, out_scale=0.01):
        self.p = dict(W1=_init(rng, i, h, math.sqrt(2)), b1=np.zeros(h), W2=_init(rng, h, h, math.sqrt(2)), b2=np.zeros(h),
                      W3=_init(rng, h, o, out_scale*math.sqrt(h)), b3=np.zeros(o))
    def forward(self, x, cache=False):
        p = self.p
        h1 = np.tanh(x @ p['W1'] + p['b1']); h2 = np.tanh(h1 @ p['W2'] + p['b2']); y = h2 @ p['W3'] + p['b3']
        return (y, (x, h1, h2)) if cache else y
    def backward(self, cache, dy):
        x, h1, h2 = cache; p = self.p; g = {}
        g['W3'] = h2.T @ dy; g['b3'] = dy.sum(0)
        d2 = (dy @ p['W3'].T)*(1 - h2*h2)
        g['W2'] = h1.T @ d2; g['b2'] = d2.sum(0)
        d1 = (d2 @ p['W2'].T)*(1 - h1*h1)
        g['W1'] = x.T @ d1; g['b1'] = d1.sum(0)
        return g

class Adam:
    def __init__(self, params, lr=3e-4, b1=0.9, b2=0.999, eps=1e-8):
        self.m = {k: np.zeros_like(v) for k, v in params.items()}; self.v = {k: np.zeros_like(v) for k, v in params.items()}
        self.t = 0; self.lr = lr; self.b1 = b1; self.b2 = b2; self.eps = eps
    def step(self, params, grads, lr=None):
        self.t += 1; lr = lr or self.lr
        for k in params:
            g = grads[k]
            self.m[k] = self.b1*self.m[k] + (1-self.b1)*g; self.v[k] = self.b2*self.v[k] + (1-self.b2)*g*g
            mh = self.m[k]/(1-self.b1**self.t); vh = self.v[k]/(1-self.b2**self.t)
            params[k] -= lr*mh/(np.sqrt(vh) + self.eps)

class Agent:
    def __init__(self, obs_dim, act_dim, seed=0, hidden=96, init_bias=-0.5, log_std=-0.7):
        rng = np.random.default_rng(seed)
        self.pi = MLP(rng, obs_dim, act_dim, hidden, 0.01); self.vf = MLP(rng, obs_dim, 1, hidden, 1.0)
        self.pi.p['b3'][:] = init_bias
        self.log_std = np.full(act_dim, log_std)
        self.norm = RunningNorm(obs_dim); self.act_dim = act_dim; self.obs_dim = obs_dim
    def act(self, obs, rng, deterministic=False):
        x = self.norm(obs); mu = self.pi.forward(x)
        if deterministic: return mu, mu, self.vf.forward(x)[:, 0]
        a = mu + np.exp(self.log_std)*rng.normal(size=mu.shape)
        logp = -0.5*(((a - mu)/np.exp(self.log_std))**2).sum(1) - self.log_std.sum() - 0.5*self.act_dim*math.log(2*math.pi)
        return a, logp, self.vf.forward(x)[:, 0]
    def value(self, obs):
        return self.vf.forward(self.norm(obs))[:, 0]
    def save(self, path):
        d = {f"pi_{k}": v for k, v in self.pi.p.items()}; d.update({f"vf_{k}": v for k, v in self.vf.p.items()})
        d.update(log_std=self.log_std, nmean=self.norm.mean, nvar=self.norm.var, nn=self.norm.n)
        np.savez(path, **d)
    def load(self, path):
        d = np.load(path)
        for k in self.pi.p: self.pi.p[k] = d[f"pi_{k}"].copy()
        for k in self.vf.p: self.vf.p[k] = d[f"vf_{k}"].copy()
        self.log_std = d['log_std'].copy(); self.norm.mean = d['nmean'].copy(); self.norm.var = d['nvar'].copy(); self.norm.n = float(d['nn'])

def bc_pretrain(agent, obs, acts, epochs=60, lr=1e-3, batch=512, seed=0, log=print):
    """supervised regression of the policy mean onto scripted-controller actions (behaviour cloning). LABEL RESULTS ACCORDINGLY."""
    rng = np.random.default_rng(seed)
    agent.norm.update(obs)
    X = agent.norm(obs); Y = acts; n = len(X); opt = Adam(agent.pi.p, lr=lr)
    for ep in range(epochs):
        perm = rng.permutation(n); tot = 0
        for i in range(0, n, batch):
            idx = perm[i:i+batch]; y, cache = agent.pi.forward(X[idx], True)
            err = y - Y[idx]; tot += (err**2).sum()
            g = agent.pi.backward(cache, 2*err/len(idx)); opt.step(agent.pi.p, g)
        if ep % 10 == 0 or ep == epochs-1: log(f"  BC epoch {ep}: mse {tot/(n*Y.shape[1]):.4f}")

def evaluate(agent, env_fn, seeds, deterministic=True):
    infos = []
    for sd in seeds:
        env = env_fn(); rng = np.random.default_rng(sd)
        o, _ = env.reset(seed=sd)
        while True:
            a, _, _ = agent.act(o[None, :], rng, deterministic)
            o, r, te, tr, info = env.step(a[0])
            if te or tr: infos.append(info); break
    return infos

def train(agent, env_fn, total_steps, n_envs=8, n_steps=256, gamma=0.99, lam=0.95, lr=3e-4, epochs=8, minibatch=1024, clip=0.2,
          ent_coef=0.0, vf_coef=0.5, rew_scale=0.1, seed=0, log=print, ckpt_path=None, ckpt_every=10, eval_fn=None, eval_every=20,
          lr_end_frac=0.2, max_grad=1.0, target_kl=0.03, start_step=0, history=None, pi_warmup=0):
    rng = np.random.default_rng(seed)
    envs = [env_fn() for _ in range(n_envs)]
    obs = np.stack([e.reset(seed=int(rng.integers(1e9)))[0] for e in envs])
    ep_ret = np.zeros(n_envs); ep_len = np.zeros(n_envs)
    popt = Adam(agent.pi.p, lr); vopt = Adam(agent.vf.p, lr)
    n_iter = max(1, total_steps // (n_envs*n_steps))
    hist = history if history is not None else []
    recent = []; t0 = time.time(); steps_done = start_step
    for it in range(n_iter):
        frac = 1 - it/n_iter; cur_lr = lr*(lr_end_frac + (1-lr_end_frac)*frac)
        B_obs = np.zeros((n_steps, n_envs, agent.obs_dim)); B_act = np.zeros((n_steps, n_envs, agent.act_dim))
        B_logp = np.zeros((n_steps, n_envs)); B_rew = np.zeros((n_steps, n_envs)); B_done = np.zeros((n_steps, n_envs))
        B_val = np.zeros((n_steps, n_envs)); B_boot = np.zeros((n_steps, n_envs))
        agent.norm.update(obs)
        for t in range(n_steps):
            a, logp, v = agent.act(obs, rng)
            B_obs[t] = obs; B_act[t] = a; B_logp[t] = logp; B_val[t] = v
            for i, e in enumerate(envs):
                o2, r, te, tr, info = e.step(a[i])
                ep_ret[i] += r; ep_len[i] += 1
                B_rew[t, i] = r*rew_scale
                if te or tr:
                    if tr and not te: B_boot[t, i] = float(agent.value(o2[None, :])[0])   # time-limit: bootstrap
                    B_done[t, i] = 1.0
                    recent.append(dict(ret=ep_ret[i], len=ep_len[i], success=info['success'], reason=info['reason'], prog=info['max_progress_deg']))
                    ep_ret[i] = 0; ep_len[i] = 0
                    o2, _ = e.reset(seed=int(rng.integers(1e9)))
                obs[i] = o2
        last_v = agent.value(obs)
        adv = np.zeros((n_steps, n_envs)); gae = 0
        for t in reversed(range(n_steps)):
            nv = last_v if t == n_steps-1 else B_val[t+1]
            nonterm = 1.0 - B_done[t]
            delta = B_rew[t] + gamma*(nv*nonterm + B_boot[t]*rew_scale) - B_val[t]
            gae = delta + gamma*lam*nonterm*gae; adv[t] = gae
        ret = adv + B_val
        N = n_steps*n_envs
        X = agent.norm(B_obs.reshape(N, -1)); A = B_act.reshape(N, -1); LP = B_logp.reshape(N); AD = adv.reshape(N); RT = ret.reshape(N)
        AD = (AD - AD.mean())/(AD.std() + 1e-8)
        kl_last = 0
        for ep in range(epochs):
            perm = rng.permutation(N); stop = False
            for i in range(0, N, minibatch):
                idx = perm[i:i+minibatch]; m = len(idx)
                mu, cache = agent.pi.forward(X[idx], True)
                std = np.exp(agent.log_std)
                logp = -0.5*(((A[idx] - mu)/std)**2).sum(1) - agent.log_std.sum() - 0.5*agent.act_dim*math.log(2*math.pi)
                ratio = np.exp(logp - LP[idx]); a_ = AD[idx]
                unclipped = (ratio*a_ < np.clip(ratio, 1-clip, 1+clip)*a_) | ((ratio >= 1-clip) & (ratio <= 1+clip))
                coef = np.where(unclipped, -a_*ratio, 0.0)/m              # dL/dlogp
                dmu = coef[:, None]*(A[idx] - mu)/std**2
                g = agent.pi.backward(cache, dmu)
                gls = (coef[:, None]*(((A[idx] - mu)/std)**2 - 1)).sum(0) - ent_coef/ m * m
                gn = math.sqrt(sum(float((v*v).sum()) for v in g.values()))
                if gn > max_grad:
                    for k in g: g[k] *= max_grad/gn
                if it >= pi_warmup:      # policy frozen during value-function warm-up (used when starting from a behaviour-cloned policy)
                    popt.step(agent.pi.p, g, cur_lr); agent.log_std = np.clip(agent.log_std - cur_lr*3*np.sign(gls)*np.minimum(np.abs(gls), 1.0), -2.5, 0.0)
                v_, vcache = agent.vf.forward(X[idx], True)
                dv = vf_coef*(v_[:, 0] - RT[idx])[:, None]/m
                gv = agent.vf.backward(vcache, dv)
                gvn = math.sqrt(sum(float((v*v).sum()) for v in gv.values()))
                if gvn > max_grad:
                    for k in gv: gv[k] *= max_grad/gvn
                vopt.step(agent.vf.p, gv, cur_lr)
                kl_last = float(np.mean(LP[idx] - logp))
            if kl_last > target_kl: break
        steps_done += N
        if (it % 5 == 0) or it == n_iter-1:
            rr = recent[-60:]
            row = dict(it=it, steps=steps_done, ret=float(np.mean([r['ret'] for r in rr])) if rr else float('nan'),
                       succ=float(np.mean([r['success'] for r in rr])) if rr else float('nan'),
                       prog=float(np.mean([r['prog'] for r in rr])) if rr else float('nan'), std=float(np.exp(agent.log_std).mean()),
                       sec=time.time()-t0)
            hist.append(row)
            log(f"it {it:4d} steps {steps_done:8d} | ret {row['ret']:7.2f} succ {row['succ']:.2f} roll {row['prog']:6.1f} deg | std {row['std']:.2f} | {row['sec']:.0f}s")
        if ckpt_path and (it % ckpt_every == 0 or it == n_iter-1):
            agent.save(ckpt_path)
        if eval_fn and (it % eval_every == 0 or it == n_iter-1):
            eval_fn(agent, steps_done)
    return hist

In [ ]:
%%writefile bc_dagger.py
import sys, time, json, numpy as np
from env3d import *
from scripted import *
from ppo_np import *
from ramp import RampExitEnv

def make_env(task, seed=None, **kw):
    return RampExitEnv(seed=seed, **kw) if task == 'ramp_exit' else RoadsterEnv(task, seed=seed, **kw)

def collect(task, n_eps, seed0, agent=None, beta=1.0, noise=0.15, env_kw=None, log=print):
    """roll out; label every visited state with the scripted controller's action. Executed action = scripted (prob beta) else learner."""
    env_kw = env_kw or {}
    O = []; A = []; rng = np.random.default_rng(seed0); succ = 0
    for ep in range(n_eps):
        env = make_env(task, seed0+ep, **env_kw); ctl = ScriptedController(env)
        o, _ = env.reset(seed=seed0+ep); ctl.reset()
        while True:
            a_s = ctl.act()
            O.append(o.copy()); A.append(a_s.copy())
            if agent is not None and rng.random() > beta:
                a_l, _, _ = agent.act(o[None, :], rng, True); a = a_l[0]
            else:
                a = a_s
            a = np.clip(a + rng.normal(0, noise, a.shape), -1, 1)
            o, r, te, tr, info = env.step(a)
            if te or tr: succ += info['success']; break
    log(f"  collected {len(O)} samples from {n_eps} episodes (executed-policy success {succ}/{n_eps})")
    return np.array(O), np.array(A)

if __name__ == "__main__":
    task = sys.argv[1]; out = sys.argv[2]
    t0 = time.time()
    def log(*a): print(*a, flush=True)
    env_fn = lambda: make_env(task)
    ag = Agent(env_fn().obs_dim, 10, seed=0, log_std=-1.8)
    O, A = collect(task, 120, 1000, noise=0.12, log=log)
    O2, A2 = collect(task, 80, 2000, noise=0.30, log=log)
    O = np.vstack([O, O2]); A = np.vstack([A, A2])
    bc_pretrain(ag, O, A, epochs=60, lr=1e-3, log=log)
    for it in range(3):
        beta = [0.5, 0.2, 0.0][it]
        On, An = collect(task, 60, 3000+100*it, agent=ag, beta=beta, noise=0.05, log=log)
        O = np.vstack([O, On]); A = np.vstack([A, An])
        bc_pretrain(ag, O, A, epochs=40, lr=5e-4, log=log)
        infos = evaluate(ag, env_fn, range(20000, 20020))
        log(f"DAgger iter {it}: dataset {len(O)}, eval success {np.mean([i['success'] for i in infos]):.2f}, roll progress {np.mean([i['max_progress_deg'] for i in infos]):.0f} deg, reasons {[(k, sum(i['reason']==k for i in infos)) for k in sorted(set(i['reason'] for i in infos))]}  [{time.time()-t0:.0f}s]")
        ag.save(out)
    np.savez(out.replace('.npz', '_data.npz'), O=O, A=A)

In [ ]:
%%writefile finetune.py
"""BC-initialised PPO fine-tuning with validation-based checkpoint selection. Reports BC-only and BC+PPO separately."""
import sys, os, json, time, numpy as np
from env3d import *
from ramp import RampExitEnv
from scripted import *
from ppo_np import *

def make_env_fn(task, **kw):
    if task == "ramp_exit": return lambda: RampExitEnv(**kw)
    return lambda: RoadsterEnv(task, **kw)

def summarize(infos):
    n = len(infos); succ = [i['success'] for i in infos]
    k = int(sum(succ)); z = 1.96; p = k/n
    den = 1 + z*z/n; c = (p + z*z/(2*n))/den; hw = z*math.sqrt(p*(1-p)/n + z*z/(4*n*n))/den
    ci = (max(0.0, c-hw), min(1.0, c+hw))
    reasons = {}
    for i in infos: reasons[i['reason']] = reasons.get(i['reason'], 0) + 1
    fails = {}
    for i in infos:
        if not i['success']:
            for kk, v in (i['checks'] or {}).items():
                if not v: fails[kk] = fails.get(kk, 0) + 1
    ok = [i for i in infos if i['success']]
    return dict(n=n, successes=k, rate=p, wilson95=ci, reasons=reasons, failed_checks=fails,
                mean_progress_deg=float(np.mean([i['max_progress_deg'] for i in infos])), mean_touch_vz=float(np.nanmean([i['touch_vz'] for i in infos])),
                mean_gas_left=float(np.mean([i['gas_left'] for i in infos])), mean_peak=float(np.mean([i['peak_height'] for i in infos])))

if __name__ == "__main__":
    task, bc_path, out_prefix, steps = sys.argv[1], sys.argv[2], sys.argv[3], int(sys.argv[4])
    def log(*a): print(*a, flush=True)
    env_fn = make_env_fn(task)
    ag = Agent(env_fn().obs_dim, 10, seed=0); ag.load(bc_path)
    val_seeds = range(20000, 20020); test_seeds = range(10000, 10100)
    res = {}
    v0 = summarize(evaluate(ag, env_fn, val_seeds)); log("BC-only validation:", v0['successes'], "/", v0['n'])
    best = dict(rate=v0['rate'], path=bc_path, steps=0)
    ag.log_std[:] = -2.0
    hist = []
    def eval_fn(agent, steps_done):
        v = summarize(evaluate(agent, env_fn, val_seeds)); log(f"   validation @ {steps_done}: {v['successes']}/{v['n']}  reasons {v['reasons']}")
        if v['rate'] >= best['rate'] and steps_done > 0:
            best.update(rate=v['rate'], path=out_prefix + "_best.npz", steps=steps_done); agent.save(out_prefix + "_best.npz")
    if steps < 0 and os.path.exists(out_prefix + '_best.npz'):      # resume-eval: training already ran, only evaluate the saved best-validation checkpoint
        best.update(rate=1.0, path=out_prefix + '_best.npz', steps=-1)
    if steps > 0:
        train(ag, env_fn, steps, n_envs=8, n_steps=256, lr=1e-4, epochs=6, ent_coef=0.0, log=log, ckpt_path=out_prefix + "_last.npz",
              eval_fn=eval_fn, eval_every=25, pi_warmup=12, history=hist, lr_end_frac=0.3)
    res['train_history'] = hist
    bc_ag = Agent(env_fn().obs_dim, 10); bc_ag.load(bc_path)
    res['test_bc_only'] = summarize(evaluate(bc_ag, env_fn, test_seeds)); log("TEST BC-only:", res['test_bc_only']['successes'], "/ 100", res['test_bc_only']['reasons'])
    if best['path'] != bc_path:
        fa = Agent(env_fn().obs_dim, 10); fa.load(best['path'])
        res['test_bc_ppo'] = summarize(evaluate(fa, env_fn, test_seeds)); res['selected'] = best
        log("TEST BC+PPO (best-validation checkpoint):", res['test_bc_ppo']['successes'], "/ 100", res['test_bc_ppo']['reasons'])
    else:
        res['test_bc_ppo'] = None; res['selected'] = best; log("PPO fine-tuning did not beat the BC policy on validation; BC policy kept")
    # scripted reference on the same test seeds
    infos = []
    for sd in test_seeds:
        env = env_fn(); ctl = ScriptedController(env); o, _ = env.reset(seed=sd); ctl.reset()
        while True:
            o, r, te, tr, info = env.step(ctl.act())
            if te or tr: infos.append(info); break
    res['test_scripted'] = summarize(infos); log("TEST scripted controller:", res['test_scripted']['successes'], "/ 100")
    json.dump(res, open(out_prefix + "_results.json", "w"), indent=1, default=float)

In [ ]:
%%writefile render3d.py
"""True 3D rendering of the SIMULATED state (position + quaternion from the physics history). Nothing is animated separately."""
import math, os, subprocess
import numpy as np
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d.art3d import Poly3DCollection, Line3DCollection
from sim3d import *
import env3d

L, W, H = 4.3, 2.0, 1.3

def box_faces(center_off, half, R, p):
    sx, sy, sz = half
    v = np.array([[x*sx, y*sy, z*sz] for x in (-1, 1) for y in (-1, 1) for z in (-1, 1)]) + center_off
    vw = p + v @ R.T
    idx = [[0, 1, 3, 2], [4, 5, 7, 6], [0, 1, 5, 4], [2, 3, 7, 6], [0, 2, 6, 4], [1, 3, 7, 5]]
    return [vw[i] for i in idx]

def wheel_poly(center_body, R, p, radius=0.33, width=0.28, n=14):
    th = np.linspace(0, 2*np.pi, n, endpoint=False)
    ring = np.stack([radius*np.cos(th), np.zeros(n), radius*np.sin(th)], 1)
    a = ring + np.array([0, -width/2, 0]); b = ring + np.array([0, width/2, 0])
    polys = []
    for i in range(n):
        j = (i+1) % n
        polys.append(np.array([a[i], a[j], b[j], b[i]]))
    polys += [a, b]
    return [p + (center_body + q) @ R.T for q in polys]


# ------------------------------------------------------------------ sports-car visual model (VISUAL ONLY: physics still uses the box + wheel contact points)
LIGHT = np.array([0.35, -0.45, 0.82]); LIGHT /= np.linalg.norm(LIGHT)
BODY = [  # x, half-width, z_bottom, z_top
    (2.15, 0.62, -0.40, -0.14), (1.95, 0.86, -0.47, -0.04), (1.40, 0.98, -0.50, 0.08), (0.70, 1.00, -0.50, 0.16),
    (-0.50, 1.00, -0.50, 0.20), (-1.50, 1.00, -0.50, 0.24), (-2.00, 0.95, -0.46, 0.27), (-2.15, 0.84, -0.42, 0.20)]
CABIN = [(0.75, 0.80, 0.16, 0.18), (0.15, 0.80, 0.18, 0.52), (-0.65, 0.78, 0.20, 0.61), (-1.25, 0.76, 0.22, 0.52), (-1.65, 0.72, 0.24, 0.27)]

def _loft(sections, colors):
    polys, cols = [], []
    for i in range(len(sections)-1):
        x0, w0, b0, t0 = sections[i]; x1, w1, b1, t1 = sections[i+1]
        top, bot_, side_c = colors[i]
        polys.append(([x0, -w0, t0], [x0, w0, t0], [x1, w1, t1], [x1, -w1, t1])); cols.append(top)
        polys.append(([x0, -w0, b0], [x0, w0, b0], [x1, w1, b1], [x1, -w1, b1])); cols.append(bot_)
        polys.append(([x0, w0, b0], [x0, w0, t0], [x1, w1, t1], [x1, w1, b1])); cols.append(side_c)
        polys.append(([x0, -w0, b0], [x0, -w0, t0], [x1, -w1, t1], [x1, -w1, b1])); cols.append(side_c)
    x, w, b, t = sections[0]; polys.append(([x, -w, b], [x, w, b], [x, w, t], [x, -w, t])); cols.append(colors[0][2])
    x, w, b, t = sections[-1]; polys.append(([x, -w, b], [x, w, b], [x, w, t], [x, -w, t])); cols.append(colors[-1][2])
    return polys, cols

def _hex(c): return np.array([int(c[1:3], 16), int(c[3:5], 16), int(c[5:7], 16)])/255.0

def draw_car(ax, R, p, cfg, body="#e8ecf2"):
    BODYC, UNDER, GLASS, ROOF, TIRE, RIM, RED, HEAD = _hex(body), _hex("#23262c"), _hex("#2b3a52"), _hex(body), _hex("#15171b"), _hex("#b8bec9"), _hex("#e5262b"), _hex("#fff6c9")
    polys, cols = [], []
    pb, cb = _loft(BODY, [(BODYC, UNDER, BODYC*0.93)]*(len(BODY)-1)); polys += pb; cols += cb
    glass = [(GLASS, UNDER, GLASS), (ROOF, UNDER, GLASS*1.1), (GLASS, UNDER, GLASS), (GLASS, UNDER, GLASS*0.9)]
    pc, cc = _loft(CABIN, glass); polys += pc; cols += cc
    def box(c, h, col):
        for q in box_faces(np.array(c, float), h, np.eye(3), np.zeros(3)): polys.append(tuple(q)); cols.append(col)
    box((-2.17, 0.0, 0.10), (0.02, 0.70, 0.045), RED)                        # tail light bar
    box((-2.17, 0.0, -0.03), (0.02, 0.30, 0.02), RED*0.7)
    box((2.0, 0.62, -0.08), (0.12, 0.2, 0.035), HEAD); box((2.0, -0.62, -0.08), (0.12, 0.2, 0.035), HEAD)   # headlights
    box((-1.95, 0.0, 0.43), (0.20, 0.82, 0.02), UNDER*1.3)                   # rear wing
    box((-1.90, 0.55, 0.35), (0.05, 0.03, 0.09), UNDER*1.3); box((-1.90, -0.55, 0.35), (0.05, 0.03, 0.09), UNDER*1.3)
    wx, wy = cfg.wheel_xy; rad = cfg.wheel_radius; wid = 0.28; nseg = 16
    th = np.linspace(0, 2*np.pi, nseg, endpoint=False)
    for sx in (1, -1):
        for sy in (1, -1):
            c = np.array([sx*wx, sy*wy, cfg.wheel_z])
            ring = lambda r, yo: np.stack([r*np.cos(th), np.full(nseg, yo), r*np.sin(th)], 1) + c
            a, b = ring(rad, -wid/2), ring(rad, wid/2)
            for i in range(nseg):
                j = (i+1) % nseg; polys.append((a[i], a[j], b[j], b[i])); cols.append(TIRE)
            polys.append(tuple(a)); cols.append(TIRE*1.2); polys.append(tuple(b)); cols.append(TIRE*1.2)
            yo = sy*(wid/2 + 0.004)                                           # silver rim on the outer face + dark hub
            polys.append(tuple(ring(rad*0.66, yo))); cols.append(RIM)
            polys.append(tuple(ring(rad*0.22, yo + sy*0.004))); cols.append(UNDER)
            for k in range(5):                                               # spokes
                ang = 2*np.pi*k/5; d = np.array([np.cos(ang), 0, np.sin(ang)])
                pp = [c + d*rad*0.2 + [0, yo + sy*0.006, 0] + np.cross(d, [0, 1, 0])*0.025, c + d*rad*0.64 + [0, yo + sy*0.006, 0] + np.cross(d, [0, 1, 0])*0.025,
                      c + d*rad*0.64 + [0, yo + sy*0.006, 0] - np.cross(d, [0, 1, 0])*0.025, c + d*rad*0.2 + [0, yo + sy*0.006, 0] - np.cross(d, [0, 1, 0])*0.025]
                polys.append(tuple(pp)); cols.append(UNDER*1.4)
    W = [p + np.array(q) @ R.T for q in polys]
    fc = []
    for q, col in zip(W, cols):
        n = np.cross(q[1] - q[0], q[2] - q[0]); nn = np.linalg.norm(n)
        sh = 0.55 + 0.45*abs(float(n @ LIGHT))/nn if nn > 1e-9 else 1.0
        fc.append(np.clip(col*sh, 0, 1))
    ax.add_collection3d(Poly3DCollection(W, facecolors=fc, edgecolors=(0.05, 0.06, 0.08, 0.55), linewidths=0.25))

def draw_frame(ax, h, cfg, nozzle_exh, nozzle_pos, fmax, obstacles, view, xlim, zlim, trail=None, title="", zoom=1.25, extra=None, show_ground=True):
    ax.cla()
    R = q_to_R(h['q']); p = h['p']
    draw_car(ax, R, p, cfg)
    # thruster plumes: along the EXHAUST direction, length proportional to thrust
    segs = []
    for i in range(len(nozzle_pos)):
        thr = h['f'][i]/fmax[i]
        if thr > 0.03:
            a = p + R @ nozzle_pos[i]; b = a + R @ nozzle_exh[i]*(0.4 + 2.4*thr)
            segs.append([a, b])
    if segs:
        ax.add_collection3d(Line3DCollection(segs, colors="#38bdf8", linewidths=3.0))
    # ground and obstacles
    gx = np.array([xlim[0], xlim[1]]); gy = np.array([-5, 5])
    if show_ground:
        ax.add_collection3d(Poly3DCollection([[[gx[0], gy[0], 0], [gx[1], gy[0], 0], [gx[1], gy[1], 0], [gx[0], gy[1], 0]]], facecolor="#2b2f36", edgecolor="#444", alpha=0.55))
    if extra is not None: extra(ax)
    for (x0, x1, hh) in obstacles:
        c = np.array([(x0+x1)/2, 0, hh/2]); half = ((x1-x0)/2, env3d.OBST_HALF_WIDTH, hh/2)
        ax.add_collection3d(Poly3DCollection(box_faces(np.zeros(3), half, np.eye(3), c), facecolor="#b0b7c3" if hh < 2 else "#e9ecef", edgecolor="#667", alpha=0.35, linewidths=0.4))
    if trail is not None and len(trail) > 1:
        tr = np.array(trail); ax.plot(tr[:, 0], tr[:, 1], tr[:, 2], color="#f59e0b", lw=1.2)
    # world axes (at the corner) and body axes (at the car)
    o = np.array([xlim[0]+0.8, -4.0, 0.0])
    for d, c, n in [((1, 0, 0), "r", "X"), ((0, 1, 0), "g", "Y"), ((0, 0, 1), "b", "Z")]:
        ax.plot([o[0], o[0]+2*d[0]], [o[1], o[1]+2*d[1]], [o[2], o[2]+2*d[2]], color=c, lw=2); ax.text(o[0]+2.2*d[0], o[1]+2.2*d[1], o[2]+2.2*d[2], n, color=c, fontsize=7)
    for k, (c, n) in enumerate([("r", "x"), ("g", "y"), ("b", "z")]):
        e = np.zeros(3); e[k] = 1.0; b = p + R @ e*2.6
        ax.plot([p[0], b[0]], [p[1], b[1]], [p[2], b[2]], color=c, lw=1.4, ls="--"); ax.text(b[0], b[1], b[2], n, color=c, fontsize=7)
    ax.set_xlim(*xlim); ax.set_ylim(-5, 5); ax.set_zlim(*zlim)
    ax.set_box_aspect((xlim[1]-xlim[0], 10, zlim[1]-zlim[0]), zoom=zoom)
    ax.view_init(elev=view[0], azim=view[1]); ax.set_title(title, fontsize=8, pad=-2)
    ax.set_facecolor("#0e1116"); ax.patch.set_facecolor("#0e1116"); ax.set_axis_off()

VIEWS = [("three-quarter", (22, -55)), ("side (looking along +Y)", (3, -90)), ("front", (3, 0)), ("rear", (3, 180))]

def render_video(hist, cfg, out_path, obstacles=(), title="", fps=25, follow=True, max_frames=None, skip=1, label_lines=None):
    nozzle_pos = np.array([n[2] for n in cfg.nozzles], float); nozzle_exh = np.array([n[3] for n in cfg.nozzles], float)
    nozzle_exh /= np.linalg.norm(nozzle_exh, axis=1, keepdims=True); fmax = np.array([n[4] for n in cfg.nozzles], float)
    fig = plt.figure(figsize=(12.8, 7.2), dpi=72)
    axs = [fig.add_subplot(2, 2, i+1, projection="3d") for i in range(4)]
    fig.subplots_adjust(left=0.0, right=1.0, top=0.90, bottom=0.0, wspace=0.0, hspace=0.0)
    frames = list(range(0, len(hist), skip))[:max_frames]
    tmp = out_path + "_frames"; os.makedirs(tmp, exist_ok=True)
    trail = []
    zmax = max(h['p'][2] for h in hist) + 3
    for n, i in enumerate(frames):
        h = hist[i]; trail.append(h['p'].copy())
        cx = h['p'][0]
        xlim = (cx-7, cx+7) if follow else (-5, 40)
        zlim = (0, max(zmax, 8))
        tilt = math.degrees(tilt_angle(h['q'])); roll = math.degrees(h.get('phi', 0))
        for ax, (nm, view) in zip(axs, VIEWS):
            draw_frame(ax, h, cfg, nozzle_exh, nozzle_pos, fmax, obstacles, view, xlim, zlim, trail=trail, title=nm)
        fig.suptitle(f"{title}\nt={h['t']:.2f}s  alt={h['p'][2]:.1f} m  tilt={tilt:.0f} deg  body-X roll progress={roll:.0f} deg  gas={h['gas']:.0f} kg  {label_lines or ''}", fontsize=9, color="w")
        fig.patch.set_facecolor("#0e1116")
        fig.savefig(os.path.join(tmp, f"f{n:04d}.png"), facecolor=fig.get_facecolor())
    plt.close(fig)
    subprocess.run(["ffmpeg", "-y", "-v", "error", "-framerate", str(fps), "-i", os.path.join(tmp, "f%04d.png"), "-pix_fmt", "yuv420p", "-vf", "pad=ceil(iw/2)*2:ceil(ih/2)*2", out_path], check=True)
    for f in os.listdir(tmp): os.remove(os.path.join(tmp, f))
    os.rmdir(tmp)
    return out_path

In [ ]:
%%writefile phase_report.py
import numpy as np, math, json, sys
from scripted import *
def phase_table(env, phases):
    H = env.hist; rows = []
    names = [n[0] for n in env.cfg.nozzles]
    grp = {g: [i for i, n in enumerate(env.cfg.nozzles) if n[1] == g] for g in ("rear", "corner", "yaw")}
    start = 0
    seq = []
    for i, ph in enumerate(phases):
        if i == 0 or ph != phases[i-1]: seq.append([ph, i, i])
        else: seq[-1][2] = i
    prev_v = np.zeros(3)
    out = []
    for ph, i0, i1 in seq:
        seg = H[i0:i1+1]
        t0 = seg[0]['t'] - 0.02; t1 = seg[-1]['t']
        z = np.array([h['p'][2] for h in seg]); v = np.array([h['v'] for h in seg]); f = np.array([h['f'] for h in seg])
        acc = (v[-1] - (H[i0-1]['v'] if i0 > 0 else v[0]))/ (t1 - t0 if t1 > t0 else 1)
        gas_u = (H[i0-1]['gas'] if i0 > 0 else env.cfg.gas0) - seg[-1]['gas']
        out.append(dict(phase=ph, t0=round(t0, 2), t1=round(t1, 2), z_start=round(float(H[i0-1]['p'][2]) if i0 > 0 else float(z[0]), 2), z_end=round(float(z[-1]), 2),
                        z_max=round(float(z.max()), 2), vz_end=round(float(v[-1][2]), 2), vx_end=round(float(v[-1][0]), 2), vy_end=round(float(v[-1][1]), 2),
                        mean_corner_kN=round(float(f[:, grp['corner']].sum(1).mean())/1e3, 1), peak_corner_kN=round(float(f[:, grp['corner']].sum(1).max())/1e3, 1),
                        mean_yaw_kN=round(float(f[:, grp['yaw']].sum(1).mean())/1e3, 2), mean_rear_kN=round(float(f[:, grp['rear']].sum(1).mean())/1e3, 2),
                        mean_acc_xyz=[round(float(a), 1) for a in acc], roll_deg_end=round(math.degrees(seg[-1]['phi']), 1), gas_kg=round(float(gas_u), 1)))
    return out
if __name__ == "__main__":
    seed = int(sys.argv[1]) if len(sys.argv) > 1 else 3
    env, info, phases = run_scripted("kickflip", seed=seed)
    for r in phase_table(env, phases): print(r)
    print({k: (round(float(v), 3) if isinstance(v, (float, np.floating)) else v) for k, v in info.items() if k not in ('checks', 'touch_rates')})
    print(info['checks'])

In [ ]:
%%writefile test_physics.py
import json, math, sys, time
import numpy as np
from sim3d import *

results = []
def check(name, ok, detail):
    results.append(dict(name=name, passed=bool(ok), detail=detail))
    print(("PASS " if ok else "FAIL ") + name + " | " + detail)

def fresh(**kw):
    s = Sim3D(VehicleConfig(**kw)); return s

# 1 ballistic
s = fresh(); s.reset(p=(1, 2, 50), v=(3, -1, 10), q=q_from_euler_zyx(0.3, 0.2, -0.4), w=(0, 0, 0))
T = 3.0
for _ in range(int(T/0.02)): s.step(np.zeros(s.nn))
pa = np.array([1 + 3*T, 2 - 1*T, 50 + 10*T - 0.5*G0*T*T]); va = np.array([3, -1, 10 - G0*T])
err = np.linalg.norm(s.p - pa); verr = np.linalg.norm(s.v - va)
check("1 zero thrust gives ballistic motion", err < 1e-6 and verr < 1e-9, f"pos err {err:.2e} m, vel err {verr:.2e} m/s over {T}s")

# 2 balanced thrusters: net torque ~ 0, force as expected
s = fresh(); names = [n[0] for n in s.cfg.nozzles]
def u_of(**d):
    u = np.zeros(s.nn)
    for k, v in d.items(): u[names.index(k)] = v
    return u
f_c = u_of(cornerFL=1, cornerFR=1, cornerRL=1, cornerRR=1) * s.fmax
F, Tq = s.wrench_body(f_c)
check("2a balanced corner nozzles: +Z force, no torque", abs(F[2] - 32000) < 1e-6 and np.abs(Tq).max() < 1e-6, f"F={F.round(1)}, tau={Tq.round(6)}")
f_r = np.array([1.0 if n.startswith('rear') else 0.0 for n in names]) * s.fmax
F, Tq = s.wrench_body(f_r)
check("2b balanced rear nozzles: +X force (plume rearward), no torque", abs(F[0] - 80000) < 1e-6 and np.abs(Tq).max() < 1e-6, f"F={F.round(1)}, tau={Tq.round(6)}")

# 3 asymmetric -> torque sign and size
fL = u_of(cornerFL=1, cornerRL=1) * s.fmax
F, Tq = s.wrench_body(fL)
check("3a left corners fire: +roll torque tau_x=+14.4 kN m", abs(Tq[0] - 14400) < 1e-6 and abs(Tq[1]) < 1e-6, f"tau={Tq.round(1)}")
fF = u_of(cornerFL=1, cornerFR=1) * s.fmax
F, Tq = s.wrench_body(fF)
check("3b front corners fire: nose-up torque tau_y<0", Tq[1] < 0 and abs(Tq[1] + 1.7*16000) < 1e-6, f"tau={Tq.round(1)}")
fY = u_of(yawL=1) * s.fmax
F, Tq = s.wrench_body(fY)
check("3c yawL fires: force -Y at the nose, tau_z<0", F[1] < 0 and Tq[2] < 0 and abs(Tq[2] + 1.9*3000) < 1e-6, f"F={F.round(1)}, tau={Tq.round(1)}")
# integrated: roll torque produces the right angular acceleration
s = fresh(gas_depletion=False); s.reset(p=(0, 0, 100), q=(1, 0, 0, 0))
u = u_of(cornerFL=1, cornerRL=1)
s.step(u, 50)  # let the valve settle (0.25 s = 8 time constants)
w_prev = s.w[0]; s.step(u, 40); s2 = s.w[0]
a_meas = (s2 - w_prev) / (40*0.005)
a_exp = 14400 / s.I0[0]
check("3d integrated roll acceleration matches tau/Ixx", abs(a_meas - a_exp)/a_exp < 0.02, f"measured {a_meas:.3f} vs expected {a_exp:.3f} rad/s^2")

# 4 quaternion normalisation under random thrust
rng = np.random.default_rng(0)
s = fresh(); s.reset(p=(0, 0, 1e4), w=(1, 2, 3))
for _ in range(5000):
    s.step(rng.random(s.nn), 4)
check("4 quaternion stays normalised", abs(np.linalg.norm(s.q) - 1) < 1e-12, f"|q|-1 = {np.linalg.norm(s.q)-1:.2e} after 5000 control steps (100 s)")

# 5 torque-free asymmetric top: angular momentum and kinetic energy
s = fresh(); s.reset(p=(0, 0, 1e5), w=(0.2, 3.0, 0.2))
L0 = s.ang_momentum_world(); E0 = 0.5*np.sum(s.inertia*s.w**2); worst_L = 0; worst_E = 0
for _ in range(500):
    s.step(np.zeros(s.nn), 4)
    worst_L = max(worst_L, np.linalg.norm(s.ang_momentum_world() - L0)/np.linalg.norm(L0))
    worst_E = max(worst_E, abs(0.5*np.sum(s.inertia*s.w**2) - E0)/E0)
check("5 torque-free tumbling conserves |L| and KE", worst_L < 1e-3 and worst_E < 1e-3, f"max rel drift L {worst_L:.1e}, KE {worst_E:.1e} over 10 s (dt=5 ms)")

# 6 a commanded roll passes intermediate orientations; unwrapped twist reaches 2*pi
s = fresh(); s.reset(p=(0, 0, 1e5), w=(4.0, 0, 0))
tilts = []; twist = 0.0; prev = 0.0
q0 = s.q.copy()
for _ in range(int(2*math.pi/4.0/0.02) + 1):
    s.step(np.zeros(s.nn), 4)
    tilts.append(math.degrees(tilt_angle(s.q)))
    qr = qmul(qconj(q0), s.q); tw = 2*math.atan2(qr[1], qr[0]); twist += wrap(tw - prev); prev = tw
check("6 a 360 deg roll passes through 90 and 180 deg tilt", max(tilts) > 170 and 80 < tilts[len(tilts)//4] < 100, f"max tilt {max(tilts):.1f} deg, tilt at quarter {tilts[len(tilts)//4]:.1f} deg, unwrapped twist {math.degrees(twist):.1f} deg")

# 7 body-fixed jet changes world direction with attitude (unit tests: upright, sideways, inverted)
names = [n[0] for n in Sim3D().cfg.nozzles]
sim = Sim3D(); f4 = np.array([0 if not n.startswith('corner') else 8000.0 for n in names])
rows = []
ok = True
for label, roll, expect in [("upright", 0, (0, 0, 1)), ("rolled +90", math.pi/2, (0, -1, 0)), ("rolled -90", -math.pi/2, (0, 1, 0)), ("inverted", math.pi, (0, 0, -1))]:
    q = q_axis_angle([1, 0, 0], roll); Fb, _ = sim.wrench_body(f4)
    Fw = q_to_R(q) @ Fb / np.linalg.norm(Fb)
    good = np.allclose(Fw, expect, atol=1e-9); ok &= good
    rows.append(f"{label}: world dir {Fw.round(2)}")
check("7a thrust direction in world for upright/sideways/inverted attitudes", ok, "; ".join(rows))
# 7b dynamic: inverted at altitude, corner nozzles accelerate the car DOWNWARD
s = fresh(gas_depletion=False); s.reset(p=(0, 0, 500), q=q_axis_angle([1, 0, 0], math.pi))
s.step(np.zeros(s.nn)+np.array([1.0 if n.startswith('corner') else 0.0 for n in names]), 100)  # 0.5 s
s.step(np.array([1.0 if n.startswith('corner') else 0.0 for n in names]), 4)
az = (s.v[2] - 0)/ s.t
check("7b inverted: up-nozzles push the car down", s.v[2] < -G0*s.t, f"v_z = {s.v[2]:.2f} m/s after {s.t:.2f}s (free fall would be {-G0*s.t:.2f})")
# 7c sideways: body-fixed corner jets accelerate the car sideways at roll +90
s = fresh(gas_depletion=False); s.reset(p=(0, 0, 500), q=q_axis_angle([1, 0, 0], math.pi/2))
for _ in range(25): s.step(np.array([1.0 if n.startswith('corner') else 0.0 for n in names]), 4)
check("7c rolled 90 deg: corner jets push the car toward world -Y", s.v[1] < -5 and abs(s.v[0]) < 1e-6, f"v = {s.v.round(2)}")

# 8 contact: drop test, energy never increases, no wheel/chassis blow-up, rest height
def drop(dt, h=3.0, q=(1, 0, 0, 0), vx=0.0):
    s = Sim3D(VehicleConfig(dt=dt)); s.reset(p=(0, 0, h), v=(vx, 0, 0), q=q)
    ns = int(round(0.02/dt)); E0 = E_prev = s.energy() + s.elastic_energy(); e_over = -1e18; lag_gain = 0.0; gain = 0.0; maxpen = 0; first_contact = None; chassis = 0
    for k in range(int(6.0/0.02)):
        s.step(np.zeros(s.nn), ns)
        E = s.energy() + s.elastic_energy(); gain = max(gain, E - E_prev); lag = s.elastic_energy() > 0 and (s.last['wheel_n'] + s.last['chassis_n']) == 0   # first step of a contact: force not yet applied
        if lag: lag_gain = max(lag_gain, E - E0)
        else: e_over = max(e_over, E - E0)
        E_prev = E
        if s.last['chassis_n']: chassis = max(chassis, s.last['max_pen_chassis'])
        if first_contact is None and s.last['wheel_n']: first_contact = s.t
    s.lag_gain = lag_gain
    return s, gain, chassis, e_over
s, gain, chassis, e_over = drop(0.005)
_, gain_h, _, _ = drop(0.0025)
rest_expected = 0.33 - 1700*G0/(4*1.0e5) + 0.55   # wheel centre height (radius - compression) + 0.55 m
check("8a landing drop: total energy (KE+PE+spring) never exceeds its initial value", e_over <= 1e-6*1700*G0*3, f"max of E(t)-E(0) once contact forces act = {e_over:.1f} J (scale m g h = {1700*G0*3:.0f} J). Separate bookkeeping artifact: the first step of a contact is detected one step late, spring energy 0.5 k pen^2 of {s.lag_gain:.1f} J ({100*s.lag_gain/(1700*G0*3):.2f}% of the drop energy) appears for that step only")
check("8b upright car comes to rest on its wheels at the expected height", abs(s.p[2] - rest_expected) < 2e-3 and np.linalg.norm(s.v) < 1e-3, f"rest z {s.p[2]:.4f} m vs expected {rest_expected:.4f} m, |v|={np.linalg.norm(s.v):.1e}")
print(f"INFO 3 m drop (7.7 m/s) is far beyond the touchdown limit; max chassis penetration {chassis:.4f} m")
s_c, g_c, chassis_c, _ = drop(0.005, h=1.34)   # touchdown at about 3 m/s (the upper end of what the task allows)
check("8c2 3 m/s touchdown does not bottom out the landing gear", chassis_c == 0.0, f"max chassis penetration {chassis_c:.4f} m")
s2, g2, chassis2, _ = drop(0.005, q=q_axis_angle([1, 0, 0], math.pi))
check("8d inverted drop is flagged as chassis contact", chassis2 > 0.0 or s2.last['chassis_n'] > 0, f"max chassis penetration {chassis2:.4f} m")
# time-step sensitivity
sA, _, _, _ = drop(0.005); sB, _, _, _ = drop(0.0025)
check("8e rest height insensitive to dt (5 ms vs 2.5 ms)", abs(sA.p[2] - sB.p[2]) < 1e-3, f"{sA.p[2]:.5f} vs {sB.p[2]:.5f} m")
s3, g3, _, e3 = drop(0.005, vx=8.0)
check("8f car rolls on after landing with forward speed (low rolling friction), no energy gain", e3 <= 1e-6*1700*G0*3 + 0.5*1700*64 and s3.v[0] > 5.5, f"vx after 6 s: {s3.v[0]:.2f} m/s (analytic rolling-resistance estimate 6.2)")

# 9 gas accounting
s = fresh(); s.reset(p=(0, 0, 1e5)); u = np.ones(s.nn)
for _ in range(50): s.step(u, 4)
expected = s.impulse/(s.cfg.isp*G0)
check("9 gas used equals impulse / (Isp g0)", abs((s.cfg.gas0 - s.gas) - expected) < 1e-9, f"used {s.cfg.gas0 - s.gas:.3f} kg vs {expected:.3f} kg")

# speed
s = fresh(); s.reset(p=(0, 0, 1e5)); t0 = time.time()
for _ in range(500): s.step(np.random.rand(s.nn), 4)
sp = 500/(time.time()-t0)
print(f"sim speed: {sp:.0f} control steps/s (4 substeps each)")
json.dump(dict(results=results, sim_steps_per_s=sp), open('physics_tests.json', 'w'), indent=1)
print(f"\n{sum(r['passed'] for r in results)}/{len(results)} passed")
sys.exit(0 if all(r['passed'] for r in results) else 1)

In [ ]:
%%writefile exploit_tests.py
import json, numpy as np, math
from env3d import *
from scripted import *

def run(controller_kw=None, env_kw=None, seed=0, policy=None, task="kickflip"):
    env = RoadsterEnv(task, seed=seed, **(env_kw or {})); ctl = ScriptedController(env)
    for k, v in (controller_kw or {}).items(): setattr(ctl, k, v)
    o, _ = env.reset(seed=seed); ctl.reset()
    for k, v in (controller_kw or {}).items(): setattr(ctl, k, v)
    while True:
        a = policy(env, o) if policy else ctl.act()
        o, r, te, tr, info = env.step(a)
        if te or tr: return info

res = {}; seeds = range(5000, 5010)
def collect(name, **kw):
    infos = [run(seed=s, **kw) for s in seeds]
    res[name] = dict(mean_return=float(np.mean([i['ret'] for i in infos])), min_return=float(np.min([i['ret'] for i in infos])), max_return=float(np.max([i['ret'] for i in infos])),
                     success=int(sum(i['success'] for i in infos)), reasons={k: sum(i['reason'] == k for i in infos) for k in set(i['reason'] for i in infos)},
                     mean_progress_deg=float(np.mean([i['max_progress_deg'] for i in infos])), mean_t=float(np.mean([i['t_end'] for i in infos])))
    r = res[name]; print(f"{name:42s} return {r['mean_return']:8.1f}  success {r['success']}/10  roll {r['mean_progress_deg']:6.1f} deg  {r['reasons']}")

collect("A reference: scripted full maneuver")
collect("B exploit: skip the rotation (stay upright)", controller_kw=dict(skip_roll=True))
collect("C exploit: premature landing (no thrust, drop)", policy=lambda env, o: -np.ones(10))
collect("D exploit: delay touchdown (hover to timeout)", controller_kw=dict(hover_alt=4.0, skip_roll=True))
collect("E exploit: hover after completing the roll", controller_kw=dict(hover_alt=4.0))
collect("F exploit: crash into the obstacle / chassis contact", policy=lambda env, o: -np.ones(10) + 0.0, env_kw=dict())
# G: dive straight into the lineup with rear nozzles on (collision seeking)
collect("G exploit: collide on purpose (rear thrust, no lift)", policy=lambda env, o: np.concatenate([np.ones(4), -np.ones(6)]))
ref = res["A reference: scripted full maneuver"]["mean_return"]
checks = {
    "full maneuver beats skipping the rotation": ref > res["B exploit: skip the rotation (stay upright)"]["mean_return"],
    "skipping the rotation never counts as success": res["B exploit: skip the rotation (stay upright)"]["success"] == 0,
    "full maneuver beats premature landing": ref > res["C exploit: premature landing (no thrust, drop)"]["mean_return"],
    "full maneuver beats hovering to timeout": ref > res["D exploit: delay touchdown (hover to timeout)"]["mean_return"],
    "hovering after the roll without landing is worse than landing": ref > res["E exploit: hover after completing the roll"]["mean_return"],
    "colliding on purpose is worse than any non-colliding failure (hover to timeout)": res["G exploit: collide on purpose (rear thrust, no lift)"]["mean_return"] < min(res["D exploit: delay touchdown (hover to timeout)"]["mean_return"], res["E exploit: hover after completing the roll"]["mean_return"]),
    "ending the episode early by crashing is never rewarded over continuing": res["F exploit: crash into the obstacle / chassis contact"]["mean_return"] < res["D exploit: delay touchdown (hover to timeout)"]["mean_return"],
    "no exploit policy succeeds": all(res[k]["success"] == 0 for k in res if k.startswith(("B", "C", "D", "F", "G"))),
}
for k, v in checks.items(): print(("PASS " if v else "FAIL ") + k)
json.dump(dict(runs=res, checks=checks), open("exploit_tests.json", "w"), indent=1)

In [ ]:
%%writefile sensitivity.py
"""Sensitivity of the kickflip to mass, inertia and actuator strength (policy trained at NOMINAL values; scripted controller for reference)."""
import json, sys, numpy as np
from env3d import *
from scripted import *
from ppo_np import *
from finetune import summarize

def cfg_with(mass=1.0, ixx=1.0, thrust=1.0, corner=1.0):
    base = VehicleConfig(); I = list(base.box_inertia())
    nz = []
    for n, g, p, e, f in base.nozzles:
        nz.append((n, g, p, e, f*thrust*(corner if g == "corner" else 1.0)))
    return VehicleConfig(m_total0=base.m_total0*mass, inertia=(I[0]*mass*ixx, I[1]*mass, I[2]*mass), nozzles=nz)

conds = [("nominal", {})] + [(f"thrust x{k}", dict(thrust=k)) for k in (0.6, 0.7, 0.8, 0.9, 1.1, 1.2)] + \
        [(f"corner nozzles x{k}", dict(corner=k)) for k in (0.6, 0.75)] + \
        [(f"mass x{k}", dict(mass=k)) for k in (0.85, 1.15)] + [(f"roll inertia x{k}", dict(ixx=k)) for k in (0.8, 1.25)]
ck = "kick_ppo_best.npz"
ag = Agent(54, 10); ag.load(ck)
seeds = range(30000, 30020); out = []
for name, kw in conds:
    cfg = cfg_with(**kw)
    pol = []; scr = []
    for sd in seeds:
        env = RoadsterEnv("kickflip", cfg=cfg); rng = np.random.default_rng(sd); o, _ = env.reset(seed=sd)
        while True:
            a, _, _ = ag.act(o[None, :], rng, True); o, r, te, tr, info = env.step(a[0])
            if te or tr: pol.append(info); break
        env = RoadsterEnv("kickflip", cfg=cfg); ctl = ScriptedController(env); o, _ = env.reset(seed=sd); ctl.reset()
        while True:
            o, r, te, tr, info = env.step(ctl.act())
            if te or tr: scr.append(info); break
    sp, ss = summarize(pol), summarize(scr)
    row = dict(condition=name, policy_success=sp['successes'], policy_reasons=sp['reasons'], scripted_success=ss['successes'], scripted_reasons=ss['reasons'], n=len(seeds))
    out.append(row); print(row, flush=True)
    json.dump(out, open("sensitivity.json", "w"), indent=1)

In [ ]:
%%writefile make_outputs.py
"""Plots, footage comparison and videos from saved checkpoints. Usage: python make_outputs.py plots|compare|video <name>"""
import sys, os, json, math, numpy as np
import matplotlib; matplotlib.use("Agg")
import matplotlib.pyplot as plt
from env3d import *
from ramp import RampExitEnv
from scripted import *
from ppo_np import *
from render3d import *
OUT = "outputs"; os.makedirs(OUT, exist_ok=True)

def env_for(task, seed, record=True):
    return RampExitEnv(seed=seed, record=record) if task == "ramp_exit" else RoadsterEnv(task, seed=seed, record=record)

def rollout_policy(task, ckpt, seed, deterministic=True):
    env = env_for(task, seed); ag = Agent(env.obs_dim, 10)
    if ckpt: ag.load(ckpt)
    rng = np.random.default_rng(seed); o, _ = env.reset(seed=seed)
    while True:
        a, _, _ = ag.act(o[None, :], rng, deterministic); o, r, te, tr, info = env.step(a[0])
        if te or tr: return env, info

def rollout_scripted(task, seed):
    env, info, _ = None, None, None
    env = env_for(task, seed); ctl = ScriptedController(env); o, _ = env.reset(seed=seed); ctl.reset()
    while True:
        o, r, te, tr, info = env.step(ctl.act())
        if te or tr: return env, info

def plot_compare(envs, labels, path, title):
    fig, axs = plt.subplots(2, 3, figsize=(15, 8)); cols = ["#2563eb", "#dc2626", "#16a34a"]
    grp = {g: [i for i, n in enumerate(envs[0].cfg.nozzles) if n[1] == g] for g in ("rear", "corner", "yaw")}
    for k, (env, lab) in enumerate(zip(envs, labels)):
        H = env.hist; t = np.array([h['t'] for h in H]); c = cols[k]
        w = np.array([h['w'] for h in H]); p = np.array([h['p'] for h in H]); f = np.array([h['f'] for h in H])
        for j, nm in enumerate("xyz"):
            axs[0, 0].plot(t, w[:, j], color=c, ls=["-", "--", ":"][j], lw=1.3, label=f"{lab} w_{nm}" if k == 0 or True else None)
        axs[0, 1].plot(t, p[:, 2], color=c, label=lab); axs[0, 1].plot(t, p[:, 0]/8.0, color=c, ls=":", lw=1)
        axs[0, 2].plot(t, [math.degrees(h['e_att']) for h in H], color=c, label=lab)
        axs[1, 0].plot(t, [math.degrees(h['phi']) for h in H], color=c, label=f"{lab} progress")
        axs[1, 0].plot(t, [math.degrees(h['th_ref']) for h in H], color="k", ls=":", lw=1)
        axs[1, 1].plot(t, f[:, grp['corner']].sum(1)/1e3, color=c, label=f"{lab} corner")
        axs[1, 1].plot(t, f[:, grp['rear']].sum(1)/1e3, color=c, ls="--", lw=1); axs[1, 1].plot(t, f[:, grp['yaw']].sum(1)/1e3, color=c, ls=":", lw=1)
        axs[1, 2].plot(t, [h['gas'] for h in H], color=c, label=lab)
    axs[0, 0].set_title("body angular velocity (rad/s)  x solid, y dashed, z dotted"); axs[0, 1].set_title("altitude of CoM (m); dotted: x/8 m")
    axs[0, 2].set_title("attitude tracking error vs reference (deg)"); axs[1, 0].set_title("signed body-X roll progress (deg); dotted = reference")
    axs[1, 1].set_title("commanded nozzle force sums (kN): corner solid, rear dashed, yaw dotted"); axs[1, 2].set_title("propellant remaining (kg)")
    for a in axs.flat: a.grid(alpha=0.3); a.set_xlabel("time (s)")
    axs[0, 1].legend(fontsize=8); axs[1, 0].legend(fontsize=8); fig.suptitle(title); fig.tight_layout(); fig.savefig(path, dpi=110); plt.close(fig)

def footage_comparison(env, path):
    from PIL import Image
    img = Image.open("k_zoom.jpg"); Wt, Ht = 640, img.size[1]//4
    idxs = [2, 4, 6, 8, 10]; times = [13.25 + i/6 for i in idxs]; targets = [0, 90, 180, 270, 330]
    H = env.hist; cfg = env.cfg
    nozzle_pos = np.array([n[2] for n in cfg.nozzles], float); nozzle_exh = np.array([n[3] for n in cfg.nozzles], float); fmax = np.array([n[4] for n in cfg.nozzles], float)
    fig = plt.figure(figsize=(18, 7.4)); 
    for k, (i, tt, tg) in enumerate(zip(idxs, times, targets)):
        r, c = divmod(i, 3); tile = img.crop((c*Wt, r*Ht, (c+1)*Wt, (r+1)*Ht))
        ax = fig.add_subplot(2, 5, k+1); ax.imshow(tile); ax.axis("off"); ax.set_title(f"footage t={tt:.2f}s\n(my visual read: roll about {tg} deg)", fontsize=9)
        j = int(np.argmin([abs(math.degrees(h['phi']) - tg) for h in H]))
        ax3 = fig.add_subplot(2, 5, 5+k+1, projection="3d")
        draw_frame(ax3, H[j], cfg, nozzle_exh, nozzle_pos, fmax, [], (4, 90), (H[j]['p'][0]-6, H[j]['p'][0]+6), (max(0, H[j]['p'][2]-5), H[j]['p'][2]+5), title=f"simulation: roll {math.degrees(H[j]['phi']):.0f} deg, t={H[j]['t']:.2f}s")
    fig.suptitle("Qualitative comparison: Tesla Kickflip footage (top, my read of the roll stage, approximate) vs simulated policy (bottom, same roll stage, camera mirrored to match travel direction)", fontsize=10)
    fig.tight_layout(); fig.savefig(path, dpi=100); plt.close(fig)

if __name__ == "__main__":
    mode = sys.argv[1]
    if mode == "plots":
        for task, ck in [("kickflip", "kick_ppo_best.npz"), ("ramp_exit", "ramp_exit_ppo_best.npz")]:
            if not os.path.exists(ck): ck = ck.replace("_ppo_best", "_bc")
            if not os.path.exists(ck): continue
            es, _ = rollout_scripted(task, 10003), None; ep, ip = rollout_policy(task, ck, 10003)
            plot_compare([es[0], ep], ["scripted", "learned"], f"{OUT}/compare_{task}.png", f"{task}: scripted controller vs learned policy, same seed 10003")
    elif mode == "compare":
        if not os.path.exists("k_zoom.jpg"): print("k_zoom.jpg (footage crops) not found, skipping the footage comparison"); sys.exit(0)
        ep, ip = rollout_policy("kickflip", "kick_ppo_best.npz" if os.path.exists("kick_ppo_best.npz") else "kick_bc.npz", 10003)
        footage_comparison(ep, f"{OUT}/footage_vs_sim.png")
    elif mode == "video":
        name = sys.argv[2]
        cfgmap = dict(baseline=("kickflip", None), landing=("landing", "landing_ppo_best.npz"), recover180=("recover180", "recover180_ppo_best.npz"),
                      kickflip=("kickflip", "kick_ppo_best.npz"), ramp=("ramp_exit", "ramp_exit_ppo_best.npz"))
        task, ck = cfgmap[name]
        if ck and not os.path.exists(ck): ck = ck.replace("_ppo_best", "_bc")
        seed = 10003
        if ck is None:
            env = env_for(task, seed); ag = Agent(env.obs_dim, 10); rng = np.random.default_rng(seed); o, _ = env.reset(seed=seed)   # untrained network
            while True:
                a, _, _ = ag.act(o[None, :], rng, True); o, r, te, tr, info = env.step(a[0])
                if te or tr: break
            title = "UNTRAINED policy (random-initialised network, half-throttle bias), Kickflip task"
        else:
            env, info = rollout_policy(task, ck, seed); title = f"Learned policy ({os.path.basename(ck)}), task {task}"
        obs_ = OBSTACLES if task == "kickflip" else []
        res = f"result: {info['reason']}, success={info['success']}"
        render_video(env.hist, env.cfg, f"{OUT}/{name}.mp4", obstacles=obs_, title=title, skip=2, fps=25, label_lines=res)
        json.dump({k: v for k, v in info.items() if k != 'checks'} | {"checks": info['checks']}, open(f"{OUT}/{name}_episode.json", "w"), indent=1, default=float)

In [ ]:
%%writefile scratch_ppo.py
"""Pure PPO from scratch (NO behaviour cloning, no scripted controller involved in training). Reported separately."""
import sys, json, numpy as np
from env3d import *
from ramp import RampExitEnv
from ppo_np import *
from finetune import summarize, make_env_fn
task, steps = sys.argv[1], int(sys.argv[2])
env_fn = make_env_fn(task)
ag = Agent(env_fn().obs_dim, 10, seed=1, log_std=-0.7)
hist = []
def log(*a): print(*a, flush=True)
train(ag, env_fn, steps, n_envs=8, n_steps=256, lr=3e-4, epochs=8, log=log, ckpt_path=f"scratch_{task}.npz", history=hist, seed=1)
infos = evaluate(ag, env_fn, range(10000, 10100))
res = dict(task=task, steps=steps, test=summarize(infos), history=hist)
log("PURE PPO TEST:", res['test']['successes'], "/ 100", res['test']['reasons'], "mean roll progress", res['test']['mean_progress_deg'])
json.dump(res, open(f"scratch_{task}_results.json", "w"), indent=1, default=float)

In [ ]:
%%writefile render_demo.py
"""Complete demonstration video: approach, launch, roll, landing, roll-out, then slow-motion replays. Rendered from the SIMULATED state."""
import sys, os, math, subprocess, json, shutil
import numpy as np
import matplotlib; matplotlib.use("Agg")
import matplotlib.pyplot as plt
from env3d import *
from ramp import RampExitEnv
from ppo_np import Agent
from sim3d import *
import render3d
from render3d import draw_frame

def make_state(h0, h1, a):
    q = (1-a)*h0['q'] + a*h1['q']
    if np.dot(h0['q'], h1['q']) < 0: q = (1-a)*h0['q'] - a*h1['q']
    q = q/np.linalg.norm(q)
    L = lambda k: (1-a)*h0[k] + a*h1[k]
    return dict(t=L('t'), p=L('p'), v=L('v'), q=q, w=L('w'), f=L('f'), phi=L('phi'), gas=L('gas'))

def sample(hist, t):
    ts = np.array([h['t'] for h in hist]); i = int(np.clip(np.searchsorted(ts, t) - 1, 0, len(hist)-2))
    a = float(np.clip((t - ts[i])/(ts[i+1]-ts[i]), 0, 1)); return make_state(hist[i], hist[i+1], a)

def approach_hist(env, seconds=1.5, mu=0.03):
    cfg = env.cfg; s = Sim3D(cfg); dec = mu*G0
    v_s = env.vx0 + dec*seconds; x_s = env.x0 - (env.vx0*seconds + 0.5*dec*seconds**2)
    s.reset(p=(x_s, 0.0, REST_Z), v=(v_s, 0, 0), q=env.q_ref0.copy())
    H = []
    for k in range(int(seconds/0.02)):
        s.step(np.zeros(s.nn), 4)
        H.append(dict(t=-seconds + (k+1)*0.02, p=s.p.copy(), v=s.v.copy(), q=s.q.copy(), w=s.w.copy(), f=s.f.copy(), phi=0.0, gas=cfg.gas0))
    return H

def run_episode(task, ckpt, seed):
    env = RampExitEnv(seed=seed, record=True) if task == "ramp_exit" else RoadsterEnv(task, seed=seed, record=True)
    ag = Agent(env.obs_dim, 10); ag.load(ckpt); rng = np.random.default_rng(seed); o, _ = env.reset(seed=seed)
    while True:
        a, _, _ = ag.act(o[None, :], rng, True); o, r, te, tr, info = env.step(a[0])
        if te or tr: return env, info

def frame_fig(W=1280, Hh=720):
    fig = plt.figure(figsize=(W/100, Hh/100), dpi=100); fig.patch.set_facecolor("#0e1116"); return fig

def text_card(path, lines, sub=None):
    fig = frame_fig(); fig.text(0.5, 0.60, lines[0], color="w", ha="center", fontsize=30, weight="bold")
    for k, l in enumerate(lines[1:]): fig.text(0.5, 0.50 - 0.07*k, l, color="#cbd5e1", ha="center", fontsize=17)
    if sub: fig.text(0.5, 0.08, sub, color="#94a3b8", ha="center", fontsize=11)
    fig.savefig(path, facecolor=fig.get_facecolor()); plt.close(fig)

def build(task, ckpt, seed, out, scene_title, phases_fn, slow_window, obstacles, cams, approach=True, rollout=1.2):
    env, info = run_episode(task, ckpt, seed)
    cfg = env.cfg; H = env.hist
    pre = approach_hist(env) if approach else []
    full = pre + H
    # roll-out after touchdown: keep simulating is not needed, episode already includes 0.6 s of settling; extend by holding state with the sim
    nozzle_pos = np.array([n[2] for n in cfg.nozzles], float); nozzle_exh = np.array([n[3] for n in cfg.nozzles], float); fmax = np.array([n[4] for n in cfg.nozzles], float)
    tmp = out + "_frames"; shutil.rmtree(tmp, ignore_errors=True); os.makedirs(tmp)
    n = [0]
    def emit(fig):
        fig.savefig(os.path.join(tmp, f"f{n[0]:05d}.png"), facecolor=fig.get_facecolor()); n[0] += 1; plt.close(fig)
    # title card (2.4 s)
    text_card(os.path.join(tmp, "card.png"), scene_title, "Simulation inspired by public coverage. Not Tesla's controller or hardware.")
    for _ in range(60): shutil.copy(os.path.join(tmp, "card.png"), os.path.join(tmp, f"f{n[0]:05d}.png")); n[0] += 1
    t_start, t_end = full[0]['t'], full[-1]['t']
    def scene(ts, cam, label_fn, trail_from=None, zlim_top=9.0, box_zoom=1.7):
        trail = []
        for t in ts:
            st = sample(full, t); trail.append(st['p'].copy())
            fig = frame_fig(); ax = fig.add_axes([0.0, 0.0, 1.0, 0.86], projection="3d")
            cx = st['p'][0]; draw_frame(ax, st, cfg, nozzle_exh, nozzle_pos, fmax, obstacles, cam['view'], (cx-cam['half_x'], cx+cam['half_x']), (0, zlim_top),
                                        trail=trail[-60:], title="", zoom=box_zoom)
            fig.text(0.5, 0.955, label_fn(st, t), color="w", ha="center", fontsize=15, weight="bold")
            fig.text(0.5, 0.915, f"{cam['name']}   |   altitude {st['p'][2]-REST_Z:4.1f} m   |   body-X roll progress {math.degrees(st['phi']):5.0f} deg   |   propellant left {st['gas']:3.0f} kg", color="#cbd5e1", ha="center", fontsize=11)
            emit(fig)
    # A: full run, real time, chase camera
    dt = 0.04
    scene(np.arange(t_start, t_end, dt), cams[0], phases_fn(env, H, "REAL TIME"))
    # B: slow motion (0.2x) of the maneuver window, two cameras
    w0, w1 = slow_window(env, H)
    for cam in cams[1:]:
        scene(np.arange(w0, w1, dt*0.2), cam, phases_fn(env, H, "SLOW MOTION 0.2x"))
    # end card
    ok = info['success']
    text_card(os.path.join(tmp, "end.png"), ["Result of this episode", f"success = {ok}   |   touchdown speed {abs(info['touch_vz']):.2f} m/s   |   peak height {info['peak_height']:.1f} m",
              "Policy: imitation-learned (behaviour cloning + DAgger) from a scripted controller.", "Pure PPO from scratch did not converge in my budget (0/100).", "All vehicle numbers and the nozzle layout are assumptions."],
              "6-DOF rigid body, 10 body-fixed nozzles, physically simulated orientation (no scripted animation).")
    for _ in range(100): shutil.copy(os.path.join(tmp, "end.png"), os.path.join(tmp, f"f{n[0]:05d}.png")); n[0] += 1
    os.remove(os.path.join(tmp, "card.png")); os.remove(os.path.join(tmp, "end.png"))
    subprocess.run(["ffmpeg", "-y", "-v", "error", "-framerate", "25", "-i", os.path.join(tmp, "f%05d.png"), "-pix_fmt", "yuv420p", "-crf", "20", out], check=True)
    shutil.rmtree(tmp); print("frames", n[0], "->", out); return info

if __name__ == "__main__":
    which = sys.argv[1]; seed = int(sys.argv[2]) if len(sys.argv) > 2 else 10003
    if which == "kickflip":
        def phases(env, H, mode):
            tr0 = next(h['t'] for h in H if math.degrees(h['phi']) > 5); tr1 = next(h['t'] for h in H if math.degrees(h['phi']) > 355); tt = env.touch_t
            def f(st, t):
                if t < 0: ph = "1  APPROACH (rolling, no thrust)"
                elif t < tr0: ph = "2  LAUNCH BURN (corner nozzles)"
                elif t < tr1: ph = "3  360 DEGREE ROLL ABOUT THE LONG AXIS"
                elif t < tt - 0.02: ph = "4  UPRIGHT, LANDING BURN"
                else: ph = "5  TOUCHDOWN AND ROLL-OUT"
                return f"Kickflip (simulation)   {ph}   [{mode}]"
            return f
        def window(env, H):
            return next(h['t'] for h in H if math.degrees(h['phi']) > 5) - 0.25, next(h['t'] for h in H if math.degrees(h['phi']) > 355) + 0.25
        cams = [dict(name="chase camera", view=(14, -58), half_x=9.0), dict(name="side camera (like the footage)", view=(3, 90), half_x=6.5), dict(name="front camera", view=(4, 0), half_x=6.5)]
        build("kickflip", "kick_ppo_best.npz", seed, "outputs/demo_kickflip_full.mp4", ["Roadster-inspired Kickflip", "360 degree roll, powered flight over a car lineup, wheels-down landing"], phases, window, OBSTACLES, cams)
    else:
        def phases(env, H, mode):
            tr1 = next(h['t'] for h in H if math.degrees(h['phi']) > 175); tt = env.touch_t
            def f(st, t):
                ph = "1  RAMP-EXIT STATE: car inverted, high up" if t < 0.15 else ("2  180 DEGREE ROLL RECOVERY" if t < tr1 else ("3  UPRIGHT, LANDING BURN" if t < tt - 0.02 else "4  TOUCHDOWN"))
                return f"Ramp recovery, airborne part only (simulation)   {ph}   [{mode}]"
            return f
        def window(env, H):
            return 0.0, next(h['t'] for h in H if math.degrees(h['phi']) > 175) + 0.3
        cams = [dict(name="chase camera", view=(12, -58), half_x=9.0), dict(name="side camera", view=(3, 90), half_x=6.5), dict(name="front camera", view=(4, 0), half_x=6.5)]
        build("ramp_exit", "ramp_exit_ppo_best.npz", seed, "outputs/demo_ramp_recovery.mp4", ["Roadster-inspired Ramp Recovery", "Airborne recovery from ramp-exit conditions (ramp contact is NOT simulated)"], phases, window, [], cams, approach=False)

In [ ]:
%%writefile render_ramp_demo.py
"""Ramp-recovery demo video (Option A): starts at the ramp LIP in the exit state. The ramp is a visual prop only, derived from the exit state."""
import sys, os, math, shutil, subprocess, json
import numpy as np
import matplotlib; matplotlib.use("Agg")
import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d.art3d import Poly3DCollection, Line3DCollection
from env3d import *
from sim3d import *
from render3d import draw_frame
from render_demo import sample, frame_fig, text_card, run_episode

def ramp_geometry(env, p0):
    cfg = env.cfg; R = q_to_R(env.q_exit)
    psi = heading(env.q_exit); u = np.array([math.cos(psi), math.sin(psi)]); v = np.array([-math.sin(psi), math.cos(psi)])
    S = p0 + R @ np.array([0, 0, cfg.wheel_z - cfg.wheel_radius])           # surface point under the car at the lip
    n = R[:, 2]; n_a = float(n[:2] @ u); n_z = float(n[2])
    Rr = S[2]/(1.0 - n_z)                                                     # radius such that the ramp base is tangent to the ground
    C = (Rr*n_a, S[2] + Rr*n_z)                                               # arc centre in plane coordinates (a along heading, z up)
    th_S = math.atan2(-n_z, -n_a); th_end = th_S - 2*math.pi if th_S > -math.pi/2 else th_S
    return dict(O=S[:2], u=u, v=v, C=C, Rr=Rr, th0=-math.pi/2, th1=th_end, S=S, n=n)

def make_extra(geo, width=5.0, N=80):
    th = np.linspace(geo['th0'], geo['th1'], N)
    a = geo['C'][0] + geo['Rr']*np.cos(th); z = geo['C'][1] + geo['Rr']*np.sin(th)
    def P(ai, zi, w): return np.array([geo['O'][0] + ai*geo['u'][0] + w*geo['v'][0], geo['O'][1] + ai*geo['u'][1] + w*geo['v'][1], zi])
    quads = [[P(a[i], z[i], -width/2), P(a[i+1], z[i+1], -width/2), P(a[i+1], z[i+1], width/2), P(a[i], z[i], width/2)] for i in range(N-1)]
    edges = [[P(a[i], z[i], -width/2), P(a[i+1], z[i+1], -width/2)] for i in range(N-1)] + [[P(a[i], z[i], width/2), P(a[i+1], z[i+1], width/2)] for i in range(N-1)]
    def extra(ax):
        ax.add_collection3d(Poly3DCollection(quads, facecolor="#9aa3ae", edgecolor="#6b7280", alpha=0.55, linewidths=0.2))
        ax.add_collection3d(Line3DCollection(edges, colors="#ff4d4d", linewidths=1.6))
    return extra

def pick_seed():
    best = None
    for sd in range(10000, 10100):
        env, info = run_episode("ramp_exit", "ramp_exit_ppo_best.npz", sd)
        if not info['success']: continue
        R = q_to_R(env.q_exit); v0 = env.hist[0]['v']; p0 = env.hist[0]['p']
        tang = R[:, 0]; nrm = R[:, 2]
        score = float(v0 @ tang) - 1.5*abs(float(v0 @ nrm)) - 2*abs(v0[1])
        if best is None or score > best[0]: best = (score, sd, float(v0 @ tang), float(v0 @ nrm))
    return best

if __name__ == "__main__":
    seed = int(sys.argv[1]) if len(sys.argv) > 1 else None
    if seed is None:
        b = pick_seed(); print("chosen seed", b); seed = b[1]
    env, info = run_episode("ramp_exit", "ramp_exit_ppo_best.npz", seed)
    cfg = env.cfg; H = env.hist
    p0 = H[0]['p'] - H[0]['v']*0.02; geo = ramp_geometry(env, p0); extra = make_extra(geo)
    print("ramp radius %.1f m, exit height %.1f m, car heading %.0f deg" % (geo['Rr'], p0[2], math.degrees(heading(env.q_exit))))
    nozzle_pos = np.array([n[2] for n in cfg.nozzles], float); nozzle_exh = np.array([n[3] for n in cfg.nozzles], float); fmax = np.array([n[4] for n in cfg.nozzles], float)
    full = [dict(H[0], t=0.0, p=p0, v=H[0]['v'])] + H
    tr1 = next(h['t'] for h in H if math.degrees(h['phi']) > 175); tt = env.touch_t; t_end = H[-1]['t']
    out = "outputs/demo_ramp_recovery.mp4"; tmp = out + "_frames"; shutil.rmtree(tmp, ignore_errors=True); os.makedirs(tmp); n = [0]
    def emit(fig): fig.savefig(os.path.join(tmp, f"f{n[0]:05d}.png"), facecolor=fig.get_facecolor()); n[0] += 1; plt.close(fig)
    def phase(t):
        if t < 0.05: return "AT THE RAMP LIP: car upside down (prescribed exit state)"
        if t < tr1: return "LEAVES THE RAMP: 180 DEGREE ROLL RECOVERY"
        if t < tt - 0.02: return "UPRIGHT: LANDING BURN"
        return "TOUCHDOWN"
    def scene(ts, cam, mode):
        trail = []
        for t in ts:
            st = sample(full, t); trail.append(st['p'].copy())
            fig = frame_fig(); ax = fig.add_axes([0.0, 0.0, 1.0, 0.86], projection="3d")
            cx = st['p'][0] if cam['follow'] else cam['cx']; cz = st['p'][2] if cam['follow'] else cam['cz']
            zlim = (cz - cam['half_z'], cz + cam['half_z']) if cam['follow'] else (0, cam['half_z']*2)
            draw_frame(ax, st, cfg, nozzle_exh, nozzle_pos, fmax, [], cam['view'], (cx - cam['half_x'], cx + cam['half_x']), zlim, trail=trail[-80:], zoom=cam['zoom'],
                       extra=extra, show_ground=(zlim[0] < 1.0))
            fig.text(0.5, 0.955, phase(t), color="w", ha="center", fontsize=16, weight="bold")
            fig.text(0.5, 0.912, f"[{mode}]  {cam['name']}  |  t = {t:4.2f} s after the lip  |  altitude {st['p'][2]-REST_Z:4.1f} m  |  roll {math.degrees(st['phi']):4.0f} deg  |  propellant {st['gas']:3.0f} kg", color="#cbd5e1", ha="center", fontsize=10)
            fig.text(0.5, 0.012, "Ramp recovery, airborne part only. The ramp is a visual prop (not simulated).", color="#94a3b8", ha="center", fontsize=9)
            emit(fig)
    text_card(os.path.join(tmp, "card.png"), ["Roadster-inspired Ramp Recovery", "Starts at the ramp lip: car upside down, then jets roll it upright and it lands on its wheels", "The ramp is a visual prop only. Ramp contact is NOT simulated."], "Simulation inspired by public coverage. Not Tesla's controller or hardware.")
    for _ in range(70): shutil.copy(os.path.join(tmp, "card.png"), os.path.join(tmp, f"f{n[0]:05d}.png")); n[0] += 1
    wide = dict(name="wide camera", view=(6, -78), follow=False, cx=float(p0[0]) - 2.0, cz=0, half_x=15.0, half_z=13.5, zoom=1.6)
    chase = dict(name="follow camera", view=(10, -60), follow=True, half_x=8.0, half_z=6.5, zoom=1.5)
    side = dict(name="side camera (like the footage)", view=(3, 90), follow=True, half_x=6.0, half_z=4.5, zoom=1.5)
    front = dict(name="front camera (sees the nose)", view=(4, 180), follow=True, half_x=6.0, half_z=4.5, zoom=1.5)
    dt = 0.04
    scene([0.0]*35, wide, "FREEZE: exit state")                      # hold on the lip
    scene(np.arange(0.0, t_end, dt), wide, "REAL TIME")
    scene(np.arange(0.0, t_end, dt), chase, "REAL TIME")
    w1 = tr1 + 0.3
    scene(np.arange(0.0, w1, dt*0.2), side, "SLOW MOTION 0.2x")
    scene(np.arange(0.0, w1, dt*0.2), front, "SLOW MOTION 0.2x")
    text_card(os.path.join(tmp, "end.png"), ["Result of this episode", f"success = {info['success']}   |   roll {info['max_progress_deg']:.0f} deg   |   touchdown speed {abs(info['touch_vz']):.2f} m/s   |   final tilt {info['final_tilt_deg']:.1f} deg",
              "Policy: imitation-learned (behaviour cloning + DAgger) from a scripted controller.", "Pure PPO from scratch did not converge in my budget.", "Exit state, ramp radius and speed (about 3 m/s) are assumptions, and the ramp itself is not simulated."], "6-DOF rigid body, 10 body-fixed nozzles, physically simulated orientation.")
    for _ in range(100): shutil.copy(os.path.join(tmp, "end.png"), os.path.join(tmp, f"f{n[0]:05d}.png")); n[0] += 1
    os.remove(os.path.join(tmp, "card.png")); os.remove(os.path.join(tmp, "end.png"))
    subprocess.run(["ffmpeg", "-y", "-v", "error", "-framerate", "25", "-i", os.path.join(tmp, "f%05d.png"), "-pix_fmt", "yuv420p", "-crf", "20", out], check=True)
    shutil.rmtree(tmp); print("frames", n[0], "->", out, "seed", seed)
    json.dump(dict(seed=seed, ramp_radius=geo['Rr'], info={k: v for k, v in info.items() if k != 'checks'}), open("outputs/demo_ramp_recovery_episode.json", "w"), indent=1, default=float)

### Shipped data (reference height profile and checkpoints, base64)

In [ ]:
import base64, os
BLOBS = {"kick_ref.npz": "UEsDBC0AAAAAAAAAIQChsvE0//////////8FABQAdC5ucHkBABAA+AUAAAAAAAD4BQAAAAAAAJNOVU1QWQEAdgB7J2Rlc2NyJzogJzxmOCcsICdmb3J0cmFuX29yZGVyJzogRmFsc2UsICdzaGFwZSc6ICgxNzUsKSwgfSAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAKAAAAAAAAAAB7FK5H4XqUP3sUrkfheqQ/uB6F61G4rj97FK5H4Xq0P5qZmZmZmbk/uB6F61G4vj/sUbgehevBP3sUrkfhesQ/CtejcD0Kxz+amZmZmZnJPylcj8L1KMw/uB6F61G4zj+kcD0K16PQP+xRuB6F69E/MzMzMzMz0z97FK5H4XrUP8P1KFyPwtU/CtejcD0K1z9SuB6F61HYP5qZmZmZmdk/4XoUrkfh2j8pXI/C9SjcP3E9CtejcN0/uB6F61G43j8AAAAAAADgP6RwPQrXo+A/SOF6FK5H4T/sUbgehevhP4/C9Shcj+I/MzMzMzMz4z/Xo3A9CtfjP3sUrkfheuQ/H4XrUbge5T/D9Shcj8LlP2dmZmZmZuY/CtejcD0K5z+uR+F6FK7nP1K4HoXrUeg/9ihcj8L16D+amZmZmZnpPz4K16NwPeo/4XoUrkfh6j+F61G4HoXrPylcj8L1KOw/zczMzMzM7D9xPQrXo3DtPxWuR+F6FO4/uB6F61G47j9cj8L1KFzvPwAAAAAAAPA/UrgehetR8D+kcD0K16PwP/YoXI/C9fA/SOF6FK5H8T+amZmZmZnxP+xRuB6F6/E/PgrXo3A98j+PwvUoXI/yP+F6FK5H4fI/MzMzMzMz8z+F61G4HoXzP9ejcD0K1/M/KVyPwvUo9D97FK5H4Xr0P83MzMzMzPQ/H4XrUbge9T9xPQrXo3D1P8P1KFyPwvU/Fa5H4XoU9j9nZmZmZmb2P7gehetRuPY/CtejcD0K9z9cj8L1KFz3P65H4XoUrvc/AAAAAAAA+D9SuB6F61H4P6RwPQrXo/g/9ihcj8L1+D9I4XoUrkf5P5qZmZmZmfk/7FG4HoXr+T8+CtejcD36P5DC9Shcj/o/4XoUrkfh+j8zMzMzMzP7P4XrUbgehfs/16NwPQrX+z8pXI/C9Sj8P3sUrkfhevw/zczMzMzM/D8fhetRuB79P3E9CtejcP0/w/UoXI/C/T8VrkfhehT+P2dmZmZmZv4/uB6F61G4/j8K16NwPQr/P1yPwvUoXP8/rkfhehSu/z8AAAAAAAAAQClcj8L1KABAUrgehetRAEB7FK5H4XoAQKRwPQrXowBAzczMzMzMAED2KFyPwvUAQB+F61G4HgFASOF6FK5HAUBxPQrXo3ABQJqZmZmZmQFAw/UoXI/CAUDsUbgehesBQBWuR+F6FAJAPgrXo3A9AkBnZmZmZmYCQI/C9ShcjwJAuB6F61G4AkDhehSuR+ECQArXo3A9CgNAMzMzMzMzA0Bcj8L1KFwDQIXrUbgehQNArkfhehSuA0DXo3A9CtcDQAAAAAAAAARAKVyPwvUoBEBSuB6F61EEQHsUrkfhegRApHA9CtejBEDNzMzMzMwEQPYoXI/C9QRAH4XrUbgeBUBI4XoUrkcFQHE9CtejcAVAmpmZmZmZBUDD9Shcj8IFQOxRuB6F6wVAFa5H4XoUBkA+CtejcD0GQGdmZmZmZgZAj8L1KFyPBkC4HoXrUbgGQOF6FK5H4QZACtejcD0KB0AzMzMzMzMHQFyPwvUoXAdAhetRuB6FB0CuR+F6FK4HQNejcD0K1wdAAAAAAAAACEApXI/C9SgIQFK4HoXrUQhAexSuR+F6CECkcD0K16MIQM3MzMzMzAhA9ihcj8L1CEAfhetRuB4JQEjhehSuRwlAcT0K16NwCUCamZmZmZkJQMP1KFyPwglA7FG4HoXrCUAVrkfhehQKQD4K16NwPQpAZ2ZmZmZmCkCQwvUoXI8KQLgehetRuApA4XoUrkfhCkAK16NwPQoLQDMzMzMzMwtAXI/C9ShcC0CF61G4HoULQK5H4XoUrgtA16NwPQrXC0BQSwMELQAAAAAAAAAhACl3Fsn//////////wUAFAB6Lm5weQEAEAD4BQAAAAAAAPgFAAAAAAAAk05VTVBZAQB2AHsnZGVzY3InOiAnPGY4JywgJ2ZvcnRyYW5fb3JkZXInOiBGYWxzZSwgJ3NoYXBlJzogKDE3NSwpLCB9ICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgIArek4eFWtPqP/W+uS9a2uo/igPYtUj56j8iOgrNqDPrPwuUNSyYhes/3rSi8/Xs6z9zrkVj/GrsP6Dp0S6gAO0/QN2OPF+u7T/aTB5FenTuP4WAWIwSU+8/7DCylBwl8D9ud0Jx+6zwP+595hsoQfE/xvKhu6Ph8T8fQELnbo7yP5bZreuJR/M/Yvb97/QM9D/SNAEIsN70P/fZtD27vPU/8UIblhan9j9qorITwp33P9xGtbe9oPg/e6m7ggmw+T8zFw51pcv6Px6fzY6R8/s/vm4I0M0n/T9a1sM4Wmj+P2YkAck2tf8/uvpfwDGHAECco/8v8DkBQIv5XrPW8gFA/m99SuWxAkDSiVr1G3cDQNva9bN6QgRA2QZPhgEUBUCRv2VssOsFQNbCOWaHyQZAjNjKc4atB0A20RiVrZcIQGqEI8r8hwlAy8/qEnR+CkAWlm5vE3sLQLDAlRuUfQxAhVceb5aDDUDWppVZBYgOQG0BSqi8hw9A3411r6xAEEBukJsxaroQQPUe06JkMRFAvwOn3iSmEUBEELTVTxkSQK910Yx8ixJAIi6V/gL9EkDjCFN0Bm4TQIaDzU2H3hNAwu+cPm9OFEBweThLmb0UQLpVzsvYKxVACLZBCf+YFUDrpxul3gQWQLCUVxpNbxZAx6/hzyLYFkAgb/8uOj8XQJVY8CJupBdAMSKVVpgHGEDGYAZzj2gYQDCHgZAlxxhA2xRG+yYjGUDmLL1gWXwZQK2282570hlA1hoE6UQlGkBm1zIuZ3QaQLPNrC6OvxpAy8Rdx2EGG0BePod9h0gbQGOt85GkhRtAeGlKYmC9G0Cbsr0MZ+8bQA6C1UZsGxxAS2F0Uy5BHEBAJ5L+eGAcQHJsoGMheRxAslYSnvyKHEB4913765UcQG1pOt7hmRxAMgRMGOWWHEBTdDo4GY0cQA6dABnJfBxAVs2tIWBmHEDWjs7KXEocQMOC1IhGKRxApvKfhaYDHEAVBa7rAdobQJhxjs7UrBtAflLmyYx8G0DQcOeUhEkbQGVBgrECFBtABqD1ajvcGkBmPvf9U6IaQFLm3qplZhpApS3CtYAoGkCt6sEEsOgZQD7eT578phlAsV4Q+W9jGUAl4/AXFR4ZQN/JiLv41hhAAvQt/SiOGECye32ftEMYQMEYb06q9xdANENZ9ReqF0CeW5g/ClsXQNp5rkqMChdAxasch6e4FkAuooe/Y2UWQF7BKTvHEBZAwjo/8Na6FUAe1KaegWMVQIbYS5eHChVAbEzNPaivFEDmnJ2allIUQPxURfUE8xNAasWzh7SQE0CT0YlohSsTQNoe2B5dwxJAX2O3kRpYEkBr1gVblekRQCJ+7uChdxFADO/rZh0CEUBH7F8mCokQQFKM0EKcDBBAy66iDGgaD0AwskX3UhcOQJNO/9SyEg1AppkdlU8PDEDNYFpnoA8LQI5SMIYZFQpACAc1rYIgCUCb19zqRDIIQJl5oC+VSgdALUbqpYtpBkCI0yaGMI8FQBHSeSWDuwRASqvL133uA0B7wfsRGCgDQDqE2pRHaAJA+9V2DwGvAUCqwLt1OPwAQJoDCi3hTwBA02TCQtxT/z+6wZ+aoxT+PwifeXT84fw/Lq4E+Mm7+z9t7rtS7qH6P9L6uaNKlPk/BejV4r6S+D8IMpjCKZ33P8Y+BYxos/Y/knOZ8lbV9T8Of/3dzgL1P8ZbtSWoO/Q/Qv7uOLh/8z9i/ruo0c7yPy8lQoPDKPI/JKqEXFiN8T942gP6U/zwP5LlTr1qdfA/5TCRcafl7z9dYk0sWvPuP1uKwmV6L+4/3vWGBJpx7T9zb4JpHMDsP1hxMdtBA+w/whB17r136z/dC1EJNibrP1BLAwQtAAAAAAAAACEA4Tdr6///////////BgAUAHZ6Lm5weQEAEAD4BQAAAAAAAPgFAAAAAAAAk05VTVBZAQB2AHsnZGVzY3InOiAnPGY4JywgJ2ZvcnRyYW5fb3JkZXInOiBGYWxzZSwgJ3NoYXBlJzogKDE3NSwpLCB9ICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgIAoAAAAAAAAAAPOLLrb34Lo/bCj8GkKY0T+gvbjU8LPbPzwh47CQEuI/3MMZ3rpb5j8mskZh7+XqP8PXCr27ke8/9sb/l2Yn8j+H5v+bX4r0P26jnIqg7/Y/FSDNXg1W+T/W3mIXFL37P8tE+K5pJP4/79CK0PNFAEAeDZoWvXkBQFyRR5uLrQJA/uhJyVzhA0BWXqZPLxUFQNaQKoMCSQZA+mYtDdZ8B0CV9wHCqbAIQI6axIt95AlAI4ORX1EYC0CK9Qs4JUwMQALZmRL5fw1AGJz+7cyzDkDOVqjJoOcPQMOKq1K6jRBAYrd4QKQnEUDhxTcujsERQAeM6Bt4WxJAoEqMCWL1EkBbpyT3S48TQGJDs+Q1KRRAUJY50h/DFEAY5bi/CV0VQKhDMq3z9hVASpqmmt2QFkDPqxaIxyoXQNUbg3WxxBdARXTsYpteGEAuKlNQhfgYQEzrmszsgBlAZ07aoLyUGUB4sOnXNz0ZQLGfAzeSrhhAl8RoBfkRGEBYDXPv330XQMelMVuz/xZAivWZYQWgFkBIB4CfiWEWQO3EB9yZOhZAlCUXwgUfFkCGWrcOaQYWQPr85+tX6xVAxtvj6jrKFUDcTuMu66AVQP1l6kaIbhVA64+7KBMzFUDHTetFA+8UQFNivFHzohRAqvAHP2VPFEBwvxZzm/QTQHkh4DiEkhNAE2N2B7QoE0BGJggZa7YSQNtHbAijOhJASZf8mSG0EUCOuV5hjiERQGbauoqJgRBA0M60MYWlD0CrCn67HCgOQD3VtpD3iAxAFaQpqc7GCkBJ4Lj1GeEIQBqCRNww2AZAX0L3n2atBEC69CyYIWMCQGZciT3X+f8/aqNhXfP++j+tzxK0UOH1P8jiWjraoPA/3Bs7xgqF5j8K58ic7UnXP/EJeV4/ypc/wlQfGQYX1L+EPc4cIH/kv5BWyoCnYO6/DvSixx7E87//kgCXefD3v+0ymRZ8sfu/PT9+0TMI/7+w9dOz2/wAwNlJt7uaRwLAVOcDIQNrA8CvuzyY/W4EwG78mssvWwXApJSnsx02BsCClC/lCAUHwBfsCPkTzAfAfvNE/BOOCMAlnNGkYkwJwFqPD9MwBwrAFmg8cQi+CsDg0yEbOHALwG5VMiseHQzAbdSxl1PEDMBWSvZavGUNwDYsyw6FAQ7AFZoUOBWYDsCanM59/CkPwGqKuV3ftw/AwgsRJzMhEMBkuLRsGGUQwMquPZrmpxDA+jQOD9rpEMDXb5UAlzURwL19HzWqjRHAisNdRsfzEcCx1auUTmkSwBlESGSo7RLATvcMmJJ6E8AwF7FbgAwUwHKYIC8dpBTAqoq4NP5CFcDTUXG/SOoVwFGTSaABmhbA+oj52hhOF8Abmpb47/sXwBxEvkQCnRjAu28LL/AiGcA6HzTkz2wZwOJQDW+scBnASrbsK60uGcAE/rirGbwYwEaZovxRLxjAcJ8we5mUF8CqBVEfrvIWwGfdh4BHTRbAV/iT9XOmFcBaGHH2V/8UwEK+bUmXWBTA/dYC7o6yE8B9ApcPdQ0TwEOGB6xqaRLAxKbcV4XGEcC7jmOo1CQRwLI/vTZlhBDA00z2n4TKD8C6cfPW7Y4OwDolpnkcVg3Ai5ujIiggDMC6J2coKu0KwO0vuhM+vQnAJjWjGIKQCMC+U06hF2cHwLXQhvkjQQbAWtc8LNEeBcDEIfwrTwAEwNyV8G3V5QLAbtgPPKXPAcA8SOI/Db4AwFvG9azeYv+/nceRlpZU/b/667diBlP7v/03q33RYvm/AtDTz+DH9r+em4+AYOvzv02sqaJ4WfK/LnNW+oQh8L+rZrIHLGnrv+K2uAppguK/bsXcGuAn1r8iQMF7OSrKv1BLAQItAy0AAAAAAAAAIQChsvE0+AUAAPgFAAAFAAAAAAAAAAAAAACAAQAAAAB0Lm5weVBLAQItAy0AAAAAAAAAIQApdxbJ+AUAAPgFAAAFAAAAAAAAAAAAAACAAS8GAAB6Lm5weVBLAQItAy0AAAAAAAAAIQDhN2vr+AUAAPgFAAAGAAAAAAAAAAAAAACAAV4MAAB2ei5ucHlQSwUGAAAAAAMAAwCaAAAAjhIAAAAA", "kick_bc.npz": "...", "kick_ppo_best.npz": "...", "landing_bc.npz": "...", "landing_ppo_best.npz": "...", "recover90_bc.npz": "...", "recover90_ppo_best.npz": "...", "recover180_bc.npz": "...", "recover180_ppo_best.npz": "...", "roll360_bc.npz": "...", "roll360_ppo_best.npz": "...", "ramp_exit_bc.npz": "...", "ramp_exit_ppo_best.npz": "..."}
for n, b in BLOBS.items():
    open(n, 'wb').write(base64.b64decode(b))
print(sorted(BLOBS))

## 1. Physics validation (20 checks)
Ballistic motion, nozzle sign convention (upright, rolled, inverted), torque signs, quaternion norm, torque-free tumbling invariants, a commanded 360 degree roll through intermediate orientations, contact energy, rest height, gas accounting.

In [ ]:
!python test_physics.py

## 2. Reference maneuver, phase by phase (scripted controller, NOT learned)
This is the feasibility demonstration for the hypothetical L1 layout: thrust commands, accelerations, altitude and propellant per phase.

In [ ]:
!python phase_report.py 3

## 3. Reward-exploit tests
Skipped rotation, premature landing, delayed touchdown (hover), hovering after the roll, and collisions must all score below the genuine maneuver and never count as success.

In [ ]:
!python exploit_tests.py

## 4. Evaluate the shipped policies on held-out seeds (10000+)

In [ ]:
import numpy as np, json, os
from env3d import *
from ramp import RampExitEnv
from ppo_np import Agent, evaluate
from finetune import summarize, make_env_fn
from scripted import ScriptedController
rows = []
for task, name in [("landing","landing"),("recover90","recover90"),("recover180","recover180"),("roll360","roll360"),("kickflip","kick"),("ramp_exit","ramp_exit")]:
    env_fn = make_env_fn(task)
    for tag, f in [("BC+DAgger", f"{name}_bc.npz"), ("BC+DAgger+PPO fine-tune", f"{name}_ppo_best.npz")]:
        if not os.path.exists(f): continue
        ag = Agent(env_fn().obs_dim, 10); ag.load(f)
        s = summarize(evaluate(ag, env_fn, range(10000, 10000+N_EVAL)))
        rows.append((task, tag, f"{s['successes']}/{s['n']}", f"[{s['wilson95'][0]:.2f}, {s['wilson95'][1]:.2f}]", s['reasons']))
import pandas as pd
print(pd.DataFrame(rows, columns=["task", "policy (learned, NOT pure PPO)", "success", "Wilson 95%", "end reasons"]).to_string(index=False))

## 5. Plots, comparison with the footage, and 3D video (simulated state)

In [ ]:
os.makedirs('outputs', exist_ok=True)
!python make_outputs.py plots
!python make_outputs.py compare
from IPython.display import Image, display
for p in ['outputs/compare_kickflip.png', 'outputs/footage_vs_sim.png']:
    if os.path.exists(p): display(Image(p))

In [ ]:
# 3D video of the learned kickflip (takes a couple of minutes). Other names: baseline, landing, recover180, ramp
!python make_outputs.py video kickflip
from IPython.display import Video
Video('outputs/kickflip.mp4', embed=True, width=900)

### Complete demonstration videos (about 30 s each: title card, real time, slow motion, result card)
Kickflip: approach, launch, 360 degree roll, landing, roll-out. Ramp recovery: starts at the ramp lip with the car upside down. The ramp is only a visual prop, ramp contact is not simulated.

In [ ]:
!python render_demo.py kickflip
!python render_ramp_demo.py 10018
from IPython.display import Video
display(Video('outputs/demo_kickflip_full.mp4', embed=True, width=900))
display(Video('outputs/demo_ramp_recovery.mp4', embed=True, width=900))

## 6. Retraining (only if RUN_MODE == 'train')

In [ ]:
if RUN_MODE == "train":
    for t, out in [("kickflip","kick"),("ramp_exit","ramp_exit"),("landing","landing"),("recover90","recover90"),("recover180","recover180"),("roll360","roll360")]:
        !python bc_dagger.py {t} {out}_bc.npz
        !python finetune.py {t} {out}_bc.npz {out}_ppo 100000
    # pure PPO from scratch, for an honest comparison (reported separately):
    !python scratch_ppo.py landing 500000

## Limitations
- Ramp contact (stage A2: tire contact on a curved ramp, friction, suction) is NOT implemented. The ramp task only starts from ramp-exit conditions (about -170 degrees of pitch, about 20 m altitude).
- Footage reading: the kickflip is a longitudinal roll (strongly supported). For the ramp, the airborne recovery appears to be a body-X roll after ramp-induced pitch, but pitch and roll could not be cleanly separated, and the roll direction was not determined.
- Obstacle sizes, approach speed (8 to 10 m/s), nozzle layout, gas Isp, valve lag, wheel and gear model are assumptions. No aerodynamic drag.
- Learned policies were obtained by imitating the scripted controller (BC+DAgger). PPO fine-tuning did not measurably improve them. Pure PPO from scratch results are reported separately.